<div style="border-left: 8px solid #04B4E3; padding: 0.25rem 0 0.25rem 1rem;">
<h1 style="color: #133C5A; margin-bottom: 0.25rem;">Expansão da Rede Federal de Educação Profissional e Economia Municipal</h1>
<p style="color: #00598E; margin: 0;"><strong>Notebook acadêmico principal</strong> · 2007–2019 · Município-ano</p>
</div>

**Pergunta de pesquisa.** A entrada em operação de unidades associadas à Fase II da expansão da Rede Federal alterou a atividade econômica dos municípios?

Este notebook apresenta a evidência construída até aqui. O período é 2007–2019, a unidade analítica é município-ano e o outcome econômico principal futuro é o pessoal ocupado assalariado (CEMPRE 708). O objetivo causal segue em avaliação; este documento não estima efeito causal.

## 1. Motivação e desenho

Na Fase II, municípios receberam unidades em anos diferentes. Trata-se, portanto, de um tratamento escalonado em um painel longitudinal. Uma etapa futura poderá avaliar um desenho de Diferenças-em-Diferenças com tratamento escalonado, possivelmente com o estimador de Callaway–Sant'Anna, somente se os gates de identificação forem atendidos.

## 2. Fontes de dados

| Fonte | Papel |
|---|---|
| MEC/SETEC | Identificação institucional da Fase II |
| INEP/Censo Escolar | Timing e atividade das unidades |
| IBGE/DTB | Existência territorial municipal |
| IBGE/CEMPRE | Atividade econômica e outcomes |
| Cadastro nacional da Rede Federal | Exposição e elegibilidade dos controles |

In [1]:
from pathlib import Path
import sys

import plotly.graph_objects as go
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import audita_populacao_causal_cempre as d12
from visualizacao_ipt import CORES_IPT, aplicar_tema_ipt, estilizar_tabela_ipt, salvar_figura_ipt

FIGURES = ROOT / 'outputs' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
painel = d12.carrega_painel_integrado()
diagnostico = d12.auditar_fase_ii(painel)
resumo_coortes = d12.resumo_por_coorte(painel)
controles_coortes = d12.auditar_controles_por_coorte(painel)
print(f'Painel carregado offline: {len(painel):,} município-ano.')

Painel carregado offline: 72,378 município-ano.


## 3. Universo e população

A construção abaixo separa exposição observada, elegibilidade estrutural de controles e candidatos principais. Os candidatos são uma população diagnóstica: **129 não é uma amostra causal final**.

In [2]:
municipios = painel.groupby('codigo_municipio_ibge', as_index=False).first()
populacao = pd.DataFrame({
    'etapa': ['Universo municipal', 'Expostos em algum momento', 'Nunca expostos', 'Controles estruturalmente elegíveis', 'Fase II', 'Candidatos principais'],
    'n_municipios': [
        municipios['codigo_municipio_ibge'].nunique(),
        municipios.loc[municipios['ever_treated'] == True, 'codigo_municipio_ibge'].nunique(),
        municipios.loc[municipios['sem_exposicao_observada_2007_2019'], 'codigo_municipio_ibge'].nunique(),
        municipios.loc[municipios['fl_elegivel_controle_candidato'], 'codigo_municipio_ibge'].nunique(),
        municipios.loc[municipios['fase_ii'] == True, 'codigo_municipio_ibge'].nunique(),
        municipios.loc[municipios['candidato_amostra_principal'] == True, 'codigo_municipio_ibge'].nunique(),
    ],
})
display(estilizar_tabela_ipt(populacao))
cores_populacao = [CORES_IPT['AZUL_ESCURO'], CORES_IPT['AZUL_MEDIO'], CORES_IPT['AZUL_CLARO'], CORES_IPT['CIANO'], CORES_IPT['AZUL_MEDIO'], CORES_IPT['AZUL_PRINCIPAL']]
fig = go.Figure(go.Bar(
    y=populacao['etapa'][::-1], x=populacao['n_municipios'][::-1], orientation='h',
    marker_color=cores_populacao[::-1], text=populacao['n_municipios'][::-1], textposition='outside',
    hovertemplate='<b>%{y}</b><br>%{x:,.0f} municípios<extra></extra>',
))
aplicar_tema_ipt(fig, titulo='Universo, exposição e populações diagnósticas')
fig.update_layout(showlegend=False)
fig.update_yaxes(showgrid=False)
fig.add_annotation(text='As categorias não formam um funil único', xref='paper', yref='paper', x=0, y=1.12, showarrow=False, font={'color': CORES_IPT['AZUL_MEDIO']})
salvar_figura_ipt(fig, FIGURES / '01_funil_populacao.png')
fig.show()

,etapa,n_municipios
0,Universo municipal,5570
1,Expostos em algum momento,147
2,Nunca expostos,4970
3,Controles estruturalmente elegíveis,4964
4,Fase II,147
5,Candidatos principais,129


### Interpretação

O universo contém todos os municípios observados no painel analítico. Os municípios nunca expostos não se confundem com os controles estruturais: estes últimos já incorporam regras de exclusão reproduzíveis. A população de 129 candidatos principais ainda poderá ser reduzida por decisões de identificação causal que não foram tomadas.

## 4. Painel CEMPRE

A camada técnica longa contém 506.870 linhas (5.570 municípios × 13 anos × 7 variáveis). O painel analítico contém apenas município-ano existentes territorialmente; 32 município-ano pré-criação territorial foram removidos desta camada, mas preservados na camada técnica. O outcome principal é o CEMPRE 708, pessoal ocupado assalariado.

In [3]:
resumo_painel = pd.DataFrame({
    'medida': ['Município-ano analítico', 'Municípios', 'Anos', 'Variáveis CEMPRE'],
    'valor': [len(painel), painel['codigo_municipio_ibge'].nunique(), f"{painel['ano'].min()}–{painel['ano'].max()}", 7],
})
display(estilizar_tabela_ipt(resumo_painel))

,medida,valor
0,Município-ano analítico,72378
1,Municípios,5570
2,Anos,2007–2019
3,Variáveis CEMPRE,7


### Interpretação

O painel tem cobertura longitudinal de 2007 a 2019 e preserva a unidade município-ano. A disponibilidade territorial é uma condição anterior à análise do outcome; por isso, a camada analítica não trata municípios ainda inexistentes como observações econômicas ausentes.

## 5. Coortes de tratamento

A coorte é o ano candidato de entrada em operação para cada município principal. Ela organiza o diagnóstico temporal, mas não produz por si só uma conclusão causal.

In [4]:
coortes = (diagnostico.loc[diagnostico['candidato_amostra_principal'] == True]
           .groupby('ano_coorte_candidata', as_index=False)
           .size().rename(columns={'size': 'n_candidatos'}))
coortes['ano_coorte_candidata'] = coortes['ano_coorte_candidata'].astype(int)
display(estilizar_tabela_ipt(coortes))
fig = go.Figure(go.Bar(
    x=coortes['ano_coorte_candidata'].astype(str), y=coortes['n_candidatos'],
    marker_color=CORES_IPT['AZUL_PRINCIPAL'], text=coortes['n_candidatos'], textposition='outside',
    hovertemplate='Coorte %{x}<br>%{y} candidatos principais<extra></extra>',
))
aplicar_tema_ipt(fig, titulo='Candidatos principais por coorte de tratamento')
fig.update_xaxes(title='Ano da coorte', type='category')
fig.update_yaxes(title='Número de municípios', rangemode='tozero')
salvar_figura_ipt(fig, FIGURES / '02_coortes_tratamento.png')
fig.show()

,ano_coorte_candidata,n_candidatos
0,2009,21
1,2010,27
2,2011,66
3,2012,13
4,2013,2


### Interpretação

Municípios de uma mesma coorte compartilham o mesmo ano candidato de tratamento. A maior concentração ocorre em 2011; as coortes de 2012 e 2013 são pequenas e exigirão cautela em qualquer avaliação posterior de heterogeneidade.

## 6. Linha do tempo

A linha do tempo abaixo situa as coortes dentro da janela observada. Para a coorte de 2009, apenas 2007 e 2008 estão disponíveis como anos pré-tratamento.

In [5]:
fig = go.Figure()
for _, linha in coortes.iterrows():
    rotulo = f"Coorte {linha['ano_coorte_candidata']}"
    fig.add_trace(go.Scatter(x=[2007, 2019], y=[rotulo, rotulo], mode='lines', line={'color': CORES_IPT['CINZA_GRADE'], 'width': 2}, hoverinfo='skip', showlegend=False))
    fig.add_trace(go.Scatter(x=[linha['ano_coorte_candidata']], y=[rotulo], mode='markers+text', marker={'color': CORES_IPT['AZUL_PRINCIPAL'], 'size': 14}, text=[f"{linha['n_candidatos']} candidatos"], textposition='top center', hovertemplate=f"{rotulo}<br>Início: %{{x}}<br>{linha['n_candidatos']} candidatos<extra></extra>", showlegend=False))
aplicar_tema_ipt(fig, titulo='Linha do tempo das coortes de tratamento')
fig.update_xaxes(title='Ano', tickmode='linear', dtick=1, range=[2006.6, 2019.4])
fig.update_yaxes(title='', showgrid=False, categoryorder='array', categoryarray=[f'Coorte {ano}' for ano in coortes['ano_coorte_candidata'][::-1]])
fig.add_annotation(x=2008, y='Coorte 2009', text='Somente 2007 e 2008<br>antes do tratamento', showarrow=True, arrowhead=2, ax=55, ay=-45, bgcolor=CORES_IPT['CINZA_FUNDO'], bordercolor=CORES_IPT['CINZA_GRADE'], font={'color': CORES_IPT['AZUL_ESCURO']})
salvar_figura_ipt(fig, FIGURES / '03_linha_tempo_coortes.png')
fig.show()

### Interpretação

A posição da coorte no início da janela de observação limita quantos anos pré-tratamento podem ser avaliados. Isso é uma propriedade do calendário do estudo, não uma evidência de efeito nem de ausência de efeito.

## 7. Suporte temporal dos tratados

O D12 exige janelas adjacentes completas: 2 pré + 3 pós requer `g-2` a `g+2`; 3 pré + 3 pós requer `g-3` a `g+2`. Todos os anos requeridos devem existir no painel e ter CEMPRE 708 numérico utilizável.

In [6]:
suporte_tratados = resumo_coortes[['ano_coorte_candidata', 'n_candidatos', 'n_elegivel_diag_2pre_3pos', 'n_elegivel_diag_3pre_3pos']].copy()
suporte_tratados['ano_coorte_candidata'] = suporte_tratados['ano_coorte_candidata'].astype(int)
display(estilizar_tabela_ipt(suporte_tratados))
fig = go.Figure()
fig.add_bar(name='2 pré + 3 pós', x=suporte_tratados['ano_coorte_candidata'].astype(str), y=suporte_tratados['n_elegivel_diag_2pre_3pos'], marker_color=CORES_IPT['AZUL_PRINCIPAL'], text=suporte_tratados['n_elegivel_diag_2pre_3pos'], textposition='outside', hovertemplate='Coorte %{x}<br>2 pré + 3 pós: %{y}<extra></extra>')
fig.add_bar(name='3 pré + 3 pós', x=suporte_tratados['ano_coorte_candidata'].astype(str), y=suporte_tratados['n_elegivel_diag_3pre_3pos'], marker_color=CORES_IPT['CIANO'], text=suporte_tratados['n_elegivel_diag_3pre_3pos'], textposition='outside', hovertemplate='Coorte %{x}<br>3 pré + 3 pós: %{y}<extra></extra>')
aplicar_tema_ipt(fig, titulo='Suporte temporal dos candidatos principais')
fig.update_layout(barmode='group')
fig.update_xaxes(title='Ano da coorte', type='category')
fig.update_yaxes(title='Número de municípios elegíveis', rangemode='tozero')
salvar_figura_ipt(fig, FIGURES / '04_suporte_temporal_tratados.png')
fig.show()

,ano_coorte_candidata,n_candidatos,n_elegivel_diag_2pre_3pos,n_elegivel_diag_3pre_3pos
0,2009,21,21,0
1,2010,27,27,27
2,2011,66,66,66
3,2012,13,13,13
4,2013,2,2,2


### Interpretação

A janela de 2 pré + 3 pós está disponível para todos os 129 candidatos. Já 3 pré + 3 pós não é possível para os 21 municípios da coorte de 2009, porque exigiria 2006, fora do painel 2007–2019. Esta limitação é de calendário, não de missing do outcome.

## 8. Disponibilidade do outcome

A tabela avalia a disponibilidade de CEMPRE 708 nas 1.677 observações dos 129 candidatos. O outcome não foi transformado e nenhum log foi aplicado.

In [7]:
candidatos = diagnostico.loc[diagnostico['candidato_amostra_principal'] == True]
disponibilidade_outcome = pd.DataFrame({
    'categoria': ['Total', 'Observado', 'Missing', 'Sigilo', 'Indisponível', 'Zero'],
    'n_observacoes': [
        candidatos['n_708_total'].sum(), candidatos['n_708_observado'].sum(), candidatos['n_708_missing'].sum(),
        candidatos['n_708_sigilo'].sum(), candidatos['n_708_indisponivel'].sum(), candidatos['n_708_zero'].sum(),
    ],
})
display(estilizar_tabela_ipt(disponibilidade_outcome))

,categoria,n_observacoes
0,Total,1677
1,Observado,1677
2,Missing,0
3,Sigilo,0
4,Indisponível,0
5,Zero,0


### Interpretação

A disponibilidade do outcome nos candidatos principais é uma verificação de qualidade de mensuração, não uma transformação analítica. Ela permite separar limitações do outcome de limitações puramente temporais do painel.

## 9. Controles por coorte

O diagnóstico aplica a mesma regra de janela adjacente aos 4.964 controles estruturalmente elegíveis. Os 4.970 municípios nunca expostos são um conjunto diferente e mais amplo.

In [8]:
controles_exibicao = controles_coortes[['ano_coorte_candidata', 'controles_2pre_3pos', 'controles_3pre_3pos']].copy()
controles_exibicao['ano_coorte_candidata'] = controles_exibicao['ano_coorte_candidata'].astype(int)
display(estilizar_tabela_ipt(controles_exibicao))
fig = go.Figure()
fig.add_bar(name='2 pré + 3 pós', x=controles_exibicao['ano_coorte_candidata'].astype(str), y=controles_exibicao['controles_2pre_3pos'], marker_color=CORES_IPT['AZUL_PRINCIPAL'], text=controles_exibicao['controles_2pre_3pos'], textposition='outside', hovertemplate='Coorte %{x}<br>2 pré + 3 pós: %{y:,.0f}<extra></extra>')
fig.add_bar(name='3 pré + 3 pós', x=controles_exibicao['ano_coorte_candidata'].astype(str), y=controles_exibicao['controles_3pre_3pos'], marker_color=CORES_IPT['CIANO'], text=controles_exibicao['controles_3pre_3pos'], textposition='outside', hovertemplate='Coorte %{x}<br>3 pré + 3 pós: %{y:,.0f}<extra></extra>')
aplicar_tema_ipt(fig, titulo='Controles estruturais com janela adjacente completa')
fig.update_layout(barmode='group')
fig.update_xaxes(title='Ano da coorte', type='category')
fig.update_yaxes(title='Número de controles', rangemode='tozero')
salvar_figura_ipt(fig, FIGURES / '05_controles_por_coorte.png')
fig.show()

,ano_coorte_candidata,controles_2pre_3pos,controles_3pre_3pos
0,2009,4964,0
1,2010,4963,4963
2,2011,4963,4963
3,2012,4963,4963
4,2013,4963,4963


### Interpretação

A única célula sigilosa do pool estrutural é o município 5003900 em 2012. Ela não afeta a coorte de 2009, cuja janela termina em 2011; afeta as coortes de 2010 a 2013, pois 2012 pertence às suas janelas adjacentes. Este diagnóstico não seleciona nem persiste uma amostra causal final.

## 10. O que já sabemos

- O painel nacional município-ano foi construído.
- O cadastro institucional foi reconstruído.
- O tratamento é escalonado.
- Há 129 candidatos principais e 4.964 controles estruturais.
- O outcome CEMPRE 708 está diagnosticado.
- O suporte temporal foi mensurado com janelas adjacentes explícitas.

## 11. O que ainda não sabemos

Ainda **não** sabemos se o desenho causal é defensável. Faltam decidir ou verificar: grupo final de comparação, antecipação, pré-tendências, comparabilidade, especificação do event study, estimador causal e robustez.

`AMOSTRA_CAUSAL_FINAL = NÃO DEFINIDA`

`DESENHO_CAUSAL_APROVADO = NÃO`

## 12. Gate de identificação causal (D13)

Os gates anteriores construíram a infraestrutura de dados: painel CEMPRE
técnico e analítico, cadastro causal institucional, integração dos dois e
auditoria do suporte temporal (D12). Nenhum desses gates avaliou se o
desenho é **defensável** para estimação causal — apenas se os dados
existem e são utilizáveis.

Este é o objetivo do **D13 — Gate de Identificação Causal**: responder, antes
de qualquer estimação, dez perguntas sobre tratamento, comparação, janela,
timing, suporte, pré-tendências e limitações. O D13 **não estima nenhum
efeito**: não roda Callaway–Sant'Anna, não calcula ATT, não faz
event-study causal, matching, propensity score, synthetic control,
regressão de efeito, TWFE causal nem bootstrap causal. É diagnóstico puro,
executado inteiramente offline sobre os artefatos já aprovados (D10/D11/D12),
sem recalculá-los.

Ao final, o notebook classifica o estado do gate — `APTO_PARA_ESPECIFICACAO`,
`REQUER_REVISAO` ou `BLOQUEADO` — mas **mesmo se apto**, mantém
`DESENHO_CAUSAL_APROVADO = NÃO` até a especificação causal ser formalmente
congelada em uma etapa futura.

## 13. O que exatamente e o tratamento?

O `CONTRATO_CAUSAL.md` define o tratamento candidato como **presenca
operacional de campus associado a Expansao Fase II no municipio** -- um
conceito institucional, nao uma leitura direta do CEMPRE. Este gate nao
redefine o tratamento a partir do comportamento do outcome economico; ele
recupera a definicao ja aprovada no cadastro causal (`D11`).

**Camadas da populacao (nao confundir):**

1. **Fase II (147 municipios)** -- populacao institucional oficial. Todos tem
   `ever_treated=true` e `pode_ser_controle=false`, sem excecao. Nem todos os
   147 sao elegiveis para a amostra causal principal.
2. **Candidato a amostra principal (129 municipios)** -- subconjunto da Fase
   II que atende, simultaneamente: status curado como `candidato_principal`
   no cadastro causal, ausencia de tratamento preexistente ao painel, uma
   `ano_coorte_candidata` definida, e elegibilidade temporal minima (2 pre +
   3 pos). Isso e uma condicao **mecanica e necessaria**, nunca suficiente --
   os 129 nao sao a amostra causal final (`CONTRATO_CAUSAL.md`, secao
   "Cadastro causal aprovado").
3. **`ano_coorte_candidata` (campo canonico da coorte)** -- o campo que
   representa a coorte candidata de cada municipio no cadastro causal. Por
   padrao (`origem_coorte='proxy_censo'`), ele e preenchido diretamente pelo
   primeiro ano observado de EPT federal ativa no Censo Escolar -- apenas uma
   **proxy anual observacional**, nao necessariamente o ano de criacao,
   autorizacao, inauguracao ou inicio das aulas. Quando existe evidencia
   institucional documentada mais forte (`origem_coorte='institucional_validada'`,
   como Sobral/CE), ela tem precedencia sobre a proxy.

**Relacao entre evento institucional, transicao e primeiro ano completo.**
O cadastro causal reserva tres campos distintos para isso --
`ano_evento_institucional`, `ano_transicao` e `primeiro_ano_completo`. A
auditoria abaixo (sem hardcode) mostra que, dos 129 candidatos principais,
**128 estao sob `origem_coorte='proxy_censo'`**: para eles, esses tres
campos vem nulos por construcao -- nao ha transicao institucional
documentada separada do primeiro ano observado no Censo, e a coorte
candidata e o proprio ano-proxy. O **unico caso** com
`origem_coorte='institucional_validada'` e **Cabo Frio/RJ (codigo
3300704)**: o IFF Cabo Frio foi inaugurado oficialmente em 05/03/2009
(fonte institucional direta do IFF), 2009 e tratado como ano parcial
(`ano_transicao=2009`, ja listado em `anos_excluir_estimacao` no cadastro
causal) e 2010 como o primeiro ano civil completo
(`primeiro_ano_completo=2010`, igual a `ano_coorte_candidata`). Este e
exatamente o caso em que o contrato ja tem uma regra inequivoca de timing --
aplicada, nao inventada aqui -- e por isso 2009 nao deve ser lido como ano
de tratamento pleno para esse municipio especifico.

In [9]:
# Confere diretamente no cadastro causal: para os 129 candidatos principais,
# os campos de transicao/evento/primeiro-ano-completo estao nulos (a coorte
# candidata E o ano-proxy do Censo, origem_coorte='proxy_censo').
candidatos_129 = painel.loc[
    (painel['fase_ii'] == True) & (painel['candidato_amostra_principal'] == True)
].groupby('codigo_municipio_ibge', as_index=False).first()

tabela_origem_coorte = candidatos_129.groupby('origem_coorte', as_index=False).agg(
    n_municipios=('codigo_municipio_ibge', 'nunique'),
    n_com_ano_transicao=('ano_transicao', lambda s: int(s.notna().sum())),
    n_com_primeiro_ano_completo=('primeiro_ano_completo', lambda s: int(s.notna().sum())),
)
display(estilizar_tabela_ipt(tabela_origem_coorte))
print(f"Municipios candidatos: {len(candidatos_129)}")
print(f"Origem da coorte candidata (unica): {sorted(candidatos_129['origem_coorte'].unique())}")

,origem_coorte,n_municipios,n_com_ano_transicao,n_com_primeiro_ano_completo
0,institucional_validada,1,1,1
1,proxy_censo,128,0,0


Municipios candidatos: 129
Origem da coorte candidata (unica): ['institucional_validada', 'proxy_censo']


## 14. Antecipacao e transicao institucional

O `CONTRATO_CAUSAL.md` (secao "Antecipacao") e o `PROTOCOLO_PRE_ANALISE.md`
(secao 6.7) sao explicitos: **a janela de antecipacao ainda nao foi
decidida**. Nao existe, nos documentos contratuais aprovados, uma regra
geral e inequivoca de quantos anos antes do primeiro ano observado (proxy
do Censo) efeitos economicos antecipados (obras, contratacao
administrativa preparatoria, expectativa de valorizacao) devem ser
excluidos da especificacao principal ou tratados como leads no
event-study.

A auditoria da secao 13 mostra que, hoje, **128 dos 129 candidatos** nao
tem `ano_transicao`/`primeiro_ano_completo` documentados individualmente
(`origem_coorte='proxy_censo'`) -- para eles nao ha, ainda, base para
diferenciar "ano de transicao" de "ano de tratamento". O **unico caso**
com essa distincao ja resolvida institucionalmente e Cabo Frio/RJ, onde o
contrato ja aplica uma regra inequivoca (2009 excluido, 2010 como coorte).
Isso e a excecao, nao a regra: nao ha uma politica geral de antecipacao
para os outros 128 candidatos, e o caso pontual de Cabo Frio nao deve ser
generalizado como se resolvesse a decisao de antecipacao para a amostra
inteira.

**Decisao pendente deste gate -- registrada, nao inventada:**

- `JANELA_ANTECIPACAO_DEFINIDA = NAO` (para a amostra em geral);
- a especificacao principal deste D13 nao cria `post` nem `event_time`;
  quando uma especificacao causal futura precisar decidir antecipacao para
  os demais candidatos, a decisao deve vir de evidencia institucional
  adicional (datas de autorizacao/obras por municipio) ou de sensibilidade
  sobre os leads do event-study -- nunca de uma regra inventada nesta etapa
  nem escolhida pelo sinal do efeito estimado (regra explicita contra
  *specification searching*, `PROTOCOLO_PRE_ANALISE.md`, secao 11).

## 15. Grupo de comparação: auditoria do pool de 4.964 controles

`fl_elegivel_controle_candidato=True` marca 4.964 municípios candidatos a
controle — nunca controles causalmente validados (`POOL_CANDIDATO_CONTROLES.md`).
A auditoria abaixo confirma empiricamente, sobre o painel integrado real,
três propriedades que o desenho precisa (sem fazer matching, sem excluir
ninguém por outcome):

In [10]:
import diagnostica_identificacao_causal as d13

COORTES_D13 = [2009, 2010, 2011, 2012, 2013]

auditoria_pool = d13.auditar_pool_controles(painel)
for chave, valor in auditoria_pool.items():
    print(f"{chave}: {valor}")

n_pool_elegivel: 4964
n_nunca_expostos: 4970
n_pool_nao_nunca_exposto: 0
codigos_pool_nao_nunca_exposto: []
n_pool_fase_ii: 0
codigos_pool_fase_ii: []
n_nunca_exposto_fora_do_pool: 6
motivos_nunca_exposto_fora_do_pool: {'universo_incompleto': 6}
pool_e_subconjunto_de_nunca_expostos: True
pool_sem_fase_ii: True


**Leitura da auditoria.** Os 4.964 elegíveis são um **subconjunto estrito**
dos 4.970 nunca expostos (`pool_e_subconjunto_de_nunca_expostos = True`) —
os dois conceitos não devem ser confundidos, e este gate não substitui
automaticamente um pelo outro. A diferença de 6 municípios (4.970 − 4.964)
não está no pool elegível porque não está presente nos 13 anos do universo
2007–2019 (`motivos_nunca_exposto_fora_do_pool = universo_incompleto`) —
são municípios criados depois de 2007 (ex.: Balneário Rincão/SC e Paraíso
das Águas/MS, ambos já documentados em D10 como criados em 2013), não uma
falha de exposição. Nenhum dos 4.964 elegíveis é município Fase II
(`pool_sem_fase_ii = True`) — nenhuma exceção institucional foi encontrada
e escondida. Portanto, para os fins deste gate:

`GRUPO_COMPARACAO_CANDIDATO = 4.964 controles estruturais elegíveis, todos
nunca expostos em 2007–2019, todos fora da Fase II` — confirmado
empiricamente, não presumido.

## 16. Janela temporal: duas especificações candidatas

Duas janelas adjacentes completas são comparadas, sem escolher pela que
preserva mais municípios:

- **Especificação candidata A — 2 pré + 3 pós**: `g-2, g-1, g, g+1, g+2`.
- **Especificação candidata B — 3 pré + 3 pós**: `g-3, g-2, g-1, g, g+1, g+2`.

A tabela e o gráfico abaixo reaproveitam a auditoria já aprovada no D12
(`resumo_coortes`, `controles_coortes`) — nenhum novo cálculo de
elegibilidade é feito aqui.

In [11]:
comparacao_janelas = resumo_coortes[[
    'ano_coorte_candidata', 'n_candidatos', 'n_elegivel_diag_2pre_3pos', 'n_elegivel_diag_3pre_3pos'
]].merge(
    controles_coortes[['ano_coorte_candidata', 'controles_2pre_3pos', 'controles_3pre_3pos']],
    on='ano_coorte_candidata',
)
comparacao_janelas['ano_coorte_candidata'] = comparacao_janelas['ano_coorte_candidata'].astype(int)
comparacao_janelas.columns = [
    'Coorte', 'N tratados', 'Tratados 2pre+3pos', 'Tratados 3pre+3pos',
    'Controles 2pre+3pos', 'Controles 3pre+3pos',
]
display(estilizar_tabela_ipt(comparacao_janelas))

total_a = int(comparacao_janelas['Tratados 2pre+3pos'].sum())
total_b = int(comparacao_janelas['Tratados 3pre+3pos'].sum())
print(f"Especificacao A (2pre+3pos): {total_a}/129 candidatos preservados.")
print(f"Especificacao B (3pre+3pos): {total_b}/129 candidatos preservados.")
print(f"Coorte 2009 sob B: {int(comparacao_janelas.loc[comparacao_janelas['Coorte'] == 2009, 'Tratados 3pre+3pos'].iloc[0])}/21 preservados.")

,Coorte,N tratados,Tratados 2pre+3pos,Tratados 3pre+3pos,Controles 2pre+3pos,Controles 3pre+3pos
0,2009,21,21,0,4964,0
1,2010,27,27,27,4963,4963
2,2011,66,66,66,4963,4963
3,2012,13,13,13,4963,4963
4,2013,2,2,2,4963,4963


Especificacao A (2pre+3pos): 129/129 candidatos preservados.
Especificacao B (3pre+3pos): 108/129 candidatos preservados.
Coorte 2009 sob B: 0/21 preservados.


**Por que a diferença não é sobre "manter mais N".** A especificação B exige
`g-3`, que para a coorte 2009 seria 2006 — fora do painel (2007–2019). Isso
é uma **limitação de calendário**, não de qualidade do outcome: os 1.677
município-ano dos 129 candidatos têm CEMPRE 708 completo (D12), então a
coorte 2009 não é excluída por dado faltante, mas por o painel simplesmente
não alcançar 2006. Adotar B como especificação principal eliminaria os 21
municípios da coorte 2009 inteiros — uma coorte real, institucionalmente
válida, não um artefato a descartar por conveniência.

## 17. Decisão provisória de janela (candidata, não congelada)

Com base exclusivamente na comparação acima — sem qualquer diagnóstico até
aqui apontando bloqueador —, a proposta deste gate é:

- **`JANELA_PRINCIPAL_CANDIDATA = 2 pré + 3 pós`** — preserva os 129/129
  candidatos e os 4.964/4.964 (coorte 2009) ou 4.963/4.963 (coortes
  2010–2013) controles estruturais, sem perder a coorte 2009 por limitação
  de calendário.
- **`JANELA_SENSIBILIDADE_CANDIDATA = 3 pré + 3 pós`** — usada como
  robustez sobre os 108/129 candidatos das coortes 2010–2013, exatamente
  porque tem mais história pré-tratamento disponível para diagnosticar
  tendências.

Isso é uma **proposta**, condicionada aos diagnósticos de pré-tendência das
seções seguintes. Se algum diagnóstico abaixo apontar evidência contrária,
esta seção deve ser revista — não confirmada por inércia.

## Identificação causal: por que Diferenças-em-Diferenças?

Comparar diretamente o outcome dos municípios tratados antes e depois do
campus chegar não isola o efeito do campus: qualquer coisa que mude ao
longo do tempo no Brasil inteiro (ciclo econômico, inflação, outra política
federal) também mudaria o outcome, mesmo sem campus nenhum.

A intuição de Diferenças-em-Diferenças (DiD) é comparar duas mudanças, não
dois níveis:

> "O quanto o tratado mudou **além** da mudança que também ocorreu no grupo
> de comparação?"

Em notação simples, com um só grupo tratado e um só grupo de controle, dois
períodos (antes/depois):

$$
DiD = \big(Y_{tratado,\,depois} - Y_{tratado,\,antes}\big) - \big(Y_{controle,\,depois} - Y_{controle,\,antes}\big)
$$

O primeiro termo (mudança do tratado) mistura o efeito do tratamento com
qualquer tendência comum. O segundo termo (mudança do controle) estima
essa tendência comum, assumindo que o controle a teria vivido igualmente
caso não fosse controle. Subtrair um do outro remove a tendência comum e
deixa (sob hipóteses, ver seção 20) o efeito atribuível ao tratamento.

**Isto é apenas intuição.** O nosso caso é mais complexo: municípios não são
tratados todos no mesmo ano — são tratados em 2009, 2010, 2011, 2012 ou
2013 (seção 19), o que a fórmula de dois grupos e dois períodos acima não
cobre diretamente.

### Tratamento escalonado: por que "tratado × pós" não basta

As coortes 2009, 2010, 2011, 2012 e 2013 (seção 5 acima) formam um
**tratamento escalonado** (*staggered adoption*): municípios diferentes são
tratados em anos diferentes, e o tratamento é absorvente — quem já foi
tratado permanece tratado nos anos seguintes.

Um modelo ingênuo de regressão com efeitos fixos de município e de ano
(TWFE — *two-way fixed effects*) e uma única variável `tratado × pós`
parece uma extensão natural do DiD de dois grupos, mas a literatura recente
(Goodman-Bacon, Callaway–Sant'Anna, Sun–Abraham, entre outros) mostrou que,
sob adoção escalonada, esse coeficiente único pode ser uma **média ponderada
com pesos negativos** de efeitos heterogêneos por coorte e por tempo desde
o tratamento — inclusive usando municípios já tratados como "controle" para
outros tratados mais tarde, contaminando a comparação. Por isso o
`CONTRATO_CAUSAL.md` já registra: **"TWFE convencional não será usado como
estimador causal principal."**

Este notebook não demonstra essa matemática — apenas registra por que ela
importa para o nosso desenho. O candidato metodológico adequado a
tratamento escalonado, mencionado no contrato e a ser estudado/implementado
em etapa futura (não aqui), é **Callaway–Sant'Anna**, que estima
$ATT(g,t)$ separadamente por coorte $g$ e tempo $t$, e só então agrega.

## A hipótese central: tendências paralelas

DiD (e suas extensões para tratamento escalonado) não exige que tratados e
controles tenham o **mesmo nível** antes do tratamento. Um município Fase II
pode ter, antes do campus chegar, um pessoal ocupado assalariado muito maior
que a mediana do pool de controles — isso por si só não invalida o desenho.

O que o DiD exige é mais sutil: que, **na ausência do tratamento**, seja
plausível que as trajetórias de tratados e controles teriam evoluído de
forma comparável (paralela) — não que partissem do mesmo lugar.

**Isso não é diretamente observável**: não existe um "mundo contrafactual
sem campus" para comparar. O que este notebook pode fazer é examinar as
trajetórias **antes** do tratamento (quando ambos os grupos ainda não foram
afetados pelo campus) como diagnóstico de plausibilidade — nunca como prova.
Pré-tendências paralelas observadas **não provam** que as tendências
pós-tratamento também seriam paralelas na ausência do tratamento; apenas
tornam essa hipótese mais ou menos plausível.

## 21. Análise de nível pré-tratamento (g-1)

Para cada coorte, a tabela compara a distribuição do CEMPRE 708 no ano
`g-1` entre os candidatos daquela coorte e o pool inteiro de 4.964/4.963
controles elegíveis (nenhum controle é excluído por outcome ou trajetória).

**Diferença de nível não reprova DiD automaticamente — é diagnóstico.** Os
municípios Fase II foram selecionados por critérios do MEC (social,
geográfico, desenvolvimentista) que, pelo DAG da seção 2 (`CONTRATO_CAUSAL.md`),
são justamente as características prévias que também afetam a trajetória
econômica — por isso é esperado, não surpreendente, que tratados e
controles tenham níveis de baseline muito diferentes.

In [12]:
baseline_niveis = d13.baseline_pre_tratamento_por_coorte(painel)
baseline_exibicao = baseline_niveis.copy()
baseline_exibicao['ano_coorte_candidata'] = baseline_exibicao['ano_coorte_candidata'].astype(int)
for col in ['media_tratados', 'mediana_tratados', 'p25_tratados', 'p75_tratados',
            'media_controles', 'mediana_controles', 'p25_controles', 'p75_controles']:
    baseline_exibicao[col] = baseline_exibicao[col].round(1)
display(estilizar_tabela_ipt(baseline_exibicao))

,ano_coorte_candidata,ano_baseline_g_menos_1,n_tratados,n_controles,media_tratados,mediana_tratados,p25_tratados,p75_tratados,media_controles,mediana_controles,p25_controles,p75_controles
0,2009,2008,21,4964,20236.300000,14970.000000,4887.000000,29158.000000,2321.900000,644.000000,315.000000,1611.800000
1,2010,2009,27,4964,12689.300000,7825.000000,2578.500000,19570.000000,2449.300000,701.500000,350.000000,1720.200000
2,2011,2010,66,4964,17514.500000,9599.500000,3574.200000,17588.800000,2641.800000,745.500000,367.000000,1831.500000
3,2012,2011,13,4964,30395.800000,21158.000000,9120.000000,37226.000000,2763.700000,789.000000,390.000000,1928.500000
4,2013,2012,2,4963,27015.500000,27015.500000,19541.200000,34489.800000,2799.900000,763.000000,373.000000,1935.000000


### 06 — Distribuição do outcome no baseline (g-1), tratados x controles

Boxplot com eixo Y logarítmico **apenas como recurso visual** (os dados
não são transformados para nenhuma análise — só a escala do eixo do
gráfico muda, para tornar visível a distribuição do pool de controles, que
vive numa escala muito menor que a dos tratados). Nenhum outlier é
removido.

In [13]:
dist_longa = d13.distribuicao_baseline_longa(painel)
dist_longa['coorte_rotulo'] = 'Coorte ' + dist_longa['ano_coorte_candidata'].astype(int).astype(str)

fig = go.Figure()
for grupo, cor in [('tratados', CORES_IPT['AZUL_PRINCIPAL']), ('controles', CORES_IPT['CIANO'])]:
    sub = dist_longa.loc[dist_longa['grupo'] == grupo]
    fig.add_trace(go.Box(
        x=sub['coorte_rotulo'], y=sub['pessoal_ocupado_assalariado'], name=grupo,
        marker_color=cor, boxpoints='outliers',
    ))
aplicar_tema_ipt(fig, titulo='Distribuicao do CEMPRE 708 no baseline (g-1) por coorte')
fig.update_layout(boxmode='group')
fig.update_xaxes(title='Coorte', categoryorder='array', categoryarray=[f'Coorte {a}' for a in sorted(dist_longa['ano_coorte_candidata'].unique())])
fig.update_yaxes(title='Pessoal ocupado assalariado (g-1, escala log)', type='log')
fig.add_annotation(text='Eixo log apenas para visualizacao; dado nao transformado na analise', xref='paper', yref='paper', x=0, y=1.1, showarrow=False, font={'color': CORES_IPT['AZUL_MEDIO'], 'size': 11})
salvar_figura_ipt(fig, FIGURES / '06_distribuicao_baseline_tratados_controles.png')
fig.show()

## 23. Pré-tendências descritivas em níveis (somente anos pré-tratamento)

Para cada coorte, a mediana do CEMPRE 708 dos candidatos daquela coorte e
do pool de controles, ano a ano, usando **exclusivamente anos anteriores a
`g`** — nenhuma observação pós-tratamento entra nesta série. Cada coorte
usa toda a história pré disponível no painel (não trava em 2 anos): a
coorte 2013 mostra 2007–2012 (6 anos), a 2009 mostra só 2007–2008 (2 anos).

Este é um diagnóstico de plausibilidade, não uma afirmação causal.

In [14]:
series_niveis = pd.concat(
    [d13.serie_pre_tendencia_niveis(painel, coorte) for coorte in COORTES_D13],
    ignore_index=True,
)
n_pre_por_coorte = {c: int(c - 2007) for c in COORTES_D13}
print('Anos pre disponiveis por coorte (nao limitado a 2):')
for c, n in n_pre_por_coorte.items():
    print(f'  Coorte {c}: {n} anos pre ({2007}-{c - 1})')

from plotly.subplots import make_subplots

fig = make_subplots(rows=2, cols=3, subplot_titles=[f'Coorte {c}' for c in COORTES_D13], shared_yaxes=False)
posicoes = [(1, 1), (1, 2), (1, 3), (2, 1), (2, 2)]
for (coorte, (linha, coluna)) in zip(COORTES_D13, posicoes):
    sub = series_niveis.loc[series_niveis['ano_coorte_candidata'] == coorte]
    for grupo, cor in [('tratados', CORES_IPT['AZUL_PRINCIPAL']), ('controles', CORES_IPT['CIANO'])]:
        sg = sub.loc[sub['grupo'] == grupo].sort_values('ano')
        fig.add_trace(
            go.Scatter(x=sg['ano'], y=sg['mediana_708'], mode='lines+markers', name=grupo,
                       line={'color': cor}, marker={'color': cor}, showlegend=(linha, coluna) == (1, 1)),
            row=linha, col=coluna,
        )
aplicar_tema_ipt(fig, titulo='Pre-tendencias em niveis (mediana CEMPRE 708) -- somente pre-tratamento')
fig.update_xaxes(dtick=1)
fig.update_layout(height=650)
salvar_figura_ipt(fig, FIGURES / '07_pre_tendencias_niveis.png')
fig.show()

Anos pre disponiveis por coorte (nao limitado a 2):
  Coorte 2009: 2 anos pre (2007-2008)
  Coorte 2010: 3 anos pre (2007-2009)
  Coorte 2011: 4 anos pre (2007-2010)
  Coorte 2012: 5 anos pre (2007-2011)
  Coorte 2013: 6 anos pre (2007-2012)


## 24. Pré-tendências normalizadas (índice, g-1 = 100)

Como tratados e controles têm níveis de baseline muito diferentes (seções
21-22), a comparação de **trajetória relativa** usa um índice: para cada
coorte e grupo, `g-1 = 100`, usando a mediana do grupo como base — **apenas
para visualização**, nunca substituindo o outcome bruto usado por qualquer
estimador futuro. Antes de normalizar, a rotina confirma que a mediana em
`g-1` é positiva em cada caso; se não fosse, o índice não seria calculado
(nenhum tratamento seria inventado).

In [15]:
indices = pd.concat(
    [d13.indice_pre_tendencia_normalizado(d13.serie_pre_tendencia_niveis(painel, c), coorte=c) for c in COORTES_D13],
    ignore_index=True,
)
indices_validos = indices.dropna(subset=['indice_708'])
print(f"Linhas com indice calculado: {len(indices_validos)}/{len(indices)} (mediana g-1 > 0 confirmada em todos os casos calculados).")

fig = make_subplots(rows=2, cols=3, subplot_titles=[f'Coorte {c}' for c in COORTES_D13], shared_yaxes=True)
for (coorte, (linha, coluna)) in zip(COORTES_D13, posicoes):
    sub = indices.loc[(indices['ano_coorte_candidata'] == coorte) & (indices['ano'] < coorte)]
    for grupo, cor in [('tratados', CORES_IPT['AZUL_PRINCIPAL']), ('controles', CORES_IPT['CIANO'])]:
        sg = sub.loc[sub['grupo'] == grupo].sort_values('ano')
        fig.add_trace(
            go.Scatter(x=sg['ano'], y=sg['indice_708'], mode='lines+markers', name=grupo,
                       line={'color': cor}, marker={'color': cor}, showlegend=(linha, coluna) == (1, 1)),
            row=linha, col=coluna,
        )
    fig.add_hline(y=100, line={'color': CORES_IPT['CINZA_GRADE'], 'dash': 'dot'}, row=linha, col=coluna)
aplicar_tema_ipt(fig, titulo='Indice de pre-tendencia (g-1=100) -- somente k<0, apenas visualizacao')
fig.update_xaxes(dtick=1)
fig.update_yaxes(title='Indice CEMPRE 708 (g-1=100)')
fig.update_layout(height=650)
salvar_figura_ipt(fig, FIGURES / '08_pre_tendencias_indice.png')
fig.show()

Linhas com indice calculado: 40/40 (mediana g-1 > 0 confirmada em todos os casos calculados).


## 26. Mudança pré-tratamento: `g-2` para `g-1`

Variação descritiva (absoluta e percentual) da mediana do CEMPRE 708 entre
os dois últimos anos antes do tratamento, tratados vs. controles. **Não é
um teste formal de tendências paralelas** — apenas mais um número
descritivo de apoio ao diagnóstico.

In [16]:
mudanca = d13.mudanca_pre_g2_g1(painel)
mudanca_exibicao = mudanca.copy()
mudanca_exibicao['variacao_absoluta'] = mudanca_exibicao['variacao_absoluta'].round(1)
mudanca_exibicao['variacao_percentual'] = mudanca_exibicao['variacao_percentual'].round(2)
display(estilizar_tabela_ipt(mudanca_exibicao))

fig = go.Figure()
for grupo, cor in [('tratados', CORES_IPT['AZUL_PRINCIPAL']), ('controles', CORES_IPT['CIANO'])]:
    sub = mudanca.loc[mudanca['grupo'] == grupo]
    fig.add_bar(name=grupo, x=sub['ano_coorte_candidata'].astype(int).astype(str), y=sub['variacao_percentual'],
                marker_color=cor, text=sub['variacao_percentual'].round(1), textposition='outside')
aplicar_tema_ipt(fig, titulo='Variacao percentual do CEMPRE 708 mediano entre g-2 e g-1')
fig.update_layout(barmode='group')
fig.update_xaxes(title='Coorte', type='category')
fig.update_yaxes(title='Variacao percentual (%)')
salvar_figura_ipt(fig, FIGURES / '09_mudanca_pre_g2_g1.png')
fig.show()

,ano_coorte_candidata,grupo,ano_g2,ano_g1,mediana_g2,mediana_g1,variacao_absoluta,variacao_percentual
0,2009,tratados,2007,2008,14627.000000,14970.000000,343.000000,2.340000
1,2009,controles,2007,2008,638.000000,644.000000,6.000000,0.940000
2,2010,tratados,2008,2009,7520.000000,7825.000000,305.000000,4.060000
3,2010,controles,2008,2009,644.000000,701.500000,57.500000,8.930000
4,2011,tratados,2009,2010,8444.000000,9599.500000,1155.500000,13.680000
5,2011,controles,2009,2010,701.500000,745.500000,44.000000,6.270000
6,2012,tratados,2010,2011,20428.000000,21158.000000,730.000000,3.570000
7,2012,controles,2010,2011,745.500000,789.000000,43.500000,5.840000
8,2013,tratados,2011,2012,27315.500000,27015.500000,-300.000000,-1.100000
9,2013,controles,2011,2012,789.000000,763.000000,-26.000000,-3.300000


### Limitação da coorte 2009

O painel começa em 2007. A coorte 2009 possui apenas **2007 e 2008** como
anos pré-tratamento — não há como observar 2006. Isso significa que existe
apenas **uma** mudança pré observável: `2007 → 2008` (mostrada acima:
+2,3% para os tratados, +0,9% para os controles). Essa é uma limitação
severa de capacidade diagnóstica — com um único ponto de variação, não é
possível avaliar se a trajetória pré-tratamento é estável ao longo de vários
anos, só entre dois.

Isso **não** é motivo para excluir a coorte 2009 automaticamente:

- a especificação principal candidata (2 pré + 3 pós, seção 17) **inclui**
  a coorte 2009 — ela tem suporte temporal completo (21/21) sob essa janela;
- a especificação de sensibilidade (3 pré + 3 pós) **não pode** incluí-la —
  exigiria 2006, fora do painel (0/21 sob essa janela, seção 16).

### Coortes 2010–2013: mais história pré disponível

Diferente da coorte 2009, as coortes seguintes têm mais anos pré
disponíveis no painel — e as figuras 07/08 acima já usam essa história
inteira (não travada em 2 anos), porque a janela `2pre+3pos` é a
especificação **candidata**, não o limite dos dados observáveis para
diagnóstico:

In [17]:
print('Anos pre efetivamente disponiveis no painel, por coorte (para diagnostico):')
for c in COORTES_D13:
    print(f'  Coorte {c}: {c - 2007} anos pre ({2007}-{c - 1})')

Anos pre efetivamente disponiveis no painel, por coorte (para diagnostico):
  Coorte 2009: 2 anos pre (2007-2008)
  Coorte 2010: 3 anos pre (2007-2009)
  Coorte 2011: 4 anos pre (2007-2010)
  Coorte 2012: 5 anos pre (2007-2011)
  Coorte 2013: 6 anos pre (2007-2012)


## 29. O que esta etapa não faz com os controles

Este gate audita e descreve o pool de 4.964 controles estruturais, mas
**não** faz nenhuma das seguintes coisas:

- matching (par a par ou por propensity score);
- exclusão de controle por nível de outcome diferente do dos tratados;
- *trimming* baseado em outcome;
- seleção individual de controle olhando sua trajetória.

Todas as tabelas e figuras acima usam o pool institucional canônico
inteiro (4.964, ou 4.963 quando a janela adjacente inclui 2012 — a única
célula sigilosa do pool, ver D12). Isso é deliberado: balanceamento e
seleção de controles pertencem a uma etapa posterior, condicionada a este
gate ter sido considerado apto.

## 30. Spillover e arranjos populacionais (diagnósticos já existentes)

`DIAGNOSTICO_SPILLOVER_FASE_II.md` e `DIAGNOSTICO_ARRANJOS_POPULACIONAIS_FASE_II.md`
já produziram, em etapas anteriores, uma base quantitativa de proximidade
geográfica (distância de Haversine entre sedes municipais) e de arranjos
populacionais entre os 4.964 candidatos a controle e os 147 municípios Fase
II. Ambos os documentos são explícitos: essas informações são
**diagnósticos**, não critérios automáticos de exclusão — o
`CONTRATO_CAUSAL.md` (seção "Spillovers") registra que nenhum raio de
distância é fixado sem justificativa substantiva (dados de deslocamento
pendular) ou análise de sensibilidade.

Este D13 não reabre essa análise espacial. Fica registrado como **ressalva
para robustez futura**: se a especificação causal for congelada, os
diagnósticos de spillover/arranjos populacionais já produzidos devem ser
revisitados para decidir se algum critério geográfico de exclusão é
justificável — não antes disso, e não automaticamente.

## 31. Tabela de gate por coorte

Combina, por coorte, o suporte temporal já auditado no D12 com uma
categoria **objetiva e descritiva** de disponibilidade de diagnóstico de
pré-tendência (nunca "boa"/"ruim" — apenas quantos períodos pré existem).

In [18]:
tabela_gate = d13.tabela_gate_por_coorte(painel)
display(estilizar_tabela_ipt(tabela_gate))

,ano_coorte_candidata,n_tratados,n_pre_disponiveis,suporte_2pre3pos,suporte_3pre3pos,controles_2pre3pos,controles_3pre3pos,observacao_pre_tendencia
0,2009,21,2,21,0,4964,0,LIMITADA_2_PERIODOS_PRE
1,2010,27,3,27,27,4963,4963,DIAGNOSTICO_PRE_DISPONIVEL
2,2011,66,4,66,66,4963,4963,DIAGNOSTICO_PRE_DISPONIVEL
3,2012,13,5,13,13,4963,4963,DIAGNOSTICO_PRE_DISPONIVEL
4,2013,2,6,2,2,4963,4963,DIAGNOSTICO_PRE_DISPONIVEL


## 32. Gate de identificação causal — classificação

Este gate **não** declara `DESENHO_CAUSAL_APROVADO = SIM` — essa decisão
pertence a uma etapa futura, depois de a especificação causal ser
formalmente congelada. O que este gate faz é classificar o estado atual em
uma de três categorias, a partir exclusivamente dos diagnósticos executados
acima:

- **`APTO_PARA_ESPECIFICACAO`** — população coerente, controles canônicos
  coerentes, timing definido (dentro do que o cadastro aprovado permite),
  nenhum bloqueador objetivo apareceu nos diagnósticos, e as ressalvas estão
  explicitadas;
- **`REQUER_REVISAO`** — há achado que precisa de decisão antes de seguir;
- **`BLOQUEADO`** — incompatibilidade estrutural grave.

**Checagem programática (sem hardcode) dos critérios de bloqueio objetivo:**

In [19]:
bloqueadores = []

if not auditoria_pool['pool_e_subconjunto_de_nunca_expostos']:
    bloqueadores.append('Pool de controles contem municipio com exposicao observada.')
if not auditoria_pool['pool_sem_fase_ii']:
    bloqueadores.append('Pool de controles contem municipio Fase II.')
if int(tabela_gate['suporte_2pre3pos'].sum()) != int(tabela_gate['n_tratados'].sum()):
    bloqueadores.append('Especificacao candidata principal (2pre+3pos) nao cobre todos os 129 candidatos.')
if int((tabela_gate['controles_2pre3pos'] == 0).sum()) > 0:
    bloqueadores.append('Alguma coorte ficou sem nenhum controle elegivel sob 2pre+3pos.')

ressalvas = [
    'Antecipacao (janela de leads) ainda NAO definida -- CONTRATO_CAUSAL.md, secao Antecipacao (pendencia registrada na secao 14).',
    'Grupo de comparacao final (never-treated vs. not-yet-treated, exclusoes adicionais por spillover) ainda em aberto -- CONTRATO_CAUSAL.md, secao Grupo de comparacao candidato.',
    'Spillover/arranjos populacionais permanecem apenas diagnosticos, nao aplicados como exclusao (secao 30).',
    'Coorte 2009 tem apenas uma mudanca pre observavel (2007->2008); nao suporta 3pre+3pos (secao 27).',
    'Uma unica celula de CEMPRE 708 sigilosa no pool estrutural (municipio 5003900, ano 2012) reduz o pool de 4964 para 4963 nas coortes 2010-2013 (D12).',
    'Baseline em nivel muito diferente entre tratados e controles (secao 21) -- diagnostico esperado pelo DAG (selecao nao aleatoria via criterios MEC), nao reprova o desenho por si so.',
]

GATE_IDENTIFICACAO = 'BLOQUEADO' if bloqueadores else 'APTO_PARA_ESPECIFICACAO'

DESENHO_CAUSAL_APROVADO = 'NAO'

print(f'GATE_IDENTIFICACAO = {GATE_IDENTIFICACAO}')
print(f'DESENHO_CAUSAL_APROVADO = {DESENHO_CAUSAL_APROVADO}')
print()
print(f'Bloqueadores objetivos encontrados: {len(bloqueadores)}')
for b in bloqueadores:
    print(f'  - {b}')
print()
print(f'Ressalvas explicitadas: {len(ressalvas)}')
for r in ressalvas:
    print(f'  - {r}')

GATE_IDENTIFICACAO = APTO_PARA_ESPECIFICACAO
DESENHO_CAUSAL_APROVADO = NAO

Bloqueadores objetivos encontrados: 0

Ressalvas explicitadas: 6
  - Antecipacao (janela de leads) ainda NAO definida -- CONTRATO_CAUSAL.md, secao Antecipacao (pendencia registrada na secao 14).
  - Grupo de comparacao final (never-treated vs. not-yet-treated, exclusoes adicionais por spillover) ainda em aberto -- CONTRATO_CAUSAL.md, secao Grupo de comparacao candidato.
  - Spillover/arranjos populacionais permanecem apenas diagnosticos, nao aplicados como exclusao (secao 30).
  - Coorte 2009 tem apenas uma mudanca pre observavel (2007->2008); nao suporta 3pre+3pos (secao 27).
  - Uma unica celula de CEMPRE 708 sigilosa no pool estrutural (municipio 5003900, ano 2012) reduz o pool de 4964 para 4963 nas coortes 2010-2013 (D12).
  - Baseline em nivel muito diferente entre tratados e controles (secao 21) -- diagnostico esperado pelo DAG (selecao nao aleatoria via criterios MEC), nao reprova o desenho por si so.


**Resultado: `GATE_IDENTIFICACAO = APTO_PARA_ESPECIFICACAO`**, com as
ressalvas listadas acima explicitamente carregadas para a próxima etapa.
Nenhum bloqueador objetivo apareceu: o pool de controles é limpo (subconjunto
de nunca expostos, sem Fase II), a especificação candidata principal
(2 pré + 3 pós) cobre 129/129 candidatos, e todas as coortes têm pelo menos
um controle elegível. `DESENHO_CAUSAL_APROVADO` permanece `NÃO` — este gate
autoriza avançar para o **desenho** da especificação, não para a
estimação.

## 33. Propostas candidatas (não congeladas)

A partir dos diagnósticos acima, as seguintes propostas ficam registradas
para a etapa de especificação — como **propostas**, não fatos consumados:

| Elemento | Proposta candidata |
|---|---|
| Grupo de comparação | 4.964 controles estruturais elegíveis, nunca tratados/expostos em 2007–2019, confirmados sem Fase II (seção 15) |
| Janela principal | 2 pré + 3 pós (seção 17) |
| Janela de sensibilidade | 3 pré + 3 pós, restrita às coortes 2010–2013 (seção 17) |
| Outcome principal | CEMPRE 708 (pessoal ocupado assalariado), em nível, sem transformação |
| Estimador futuro | Callaway–Sant'Anna / DiD para tratamento escalonado (não implementado aqui) |

Nenhum estimador foi executado. Nenhum efeito foi calculado.

## 34. O que este gate confirma e o que ainda falta

**Confirmado neste D13:**

- o tratamento é institucional (proxy do Censo), não redefinido pelo CEMPRE;
- o pool de 4.964 controles é limpo (nunca expostos, sem Fase II);
- a especificação 2 pré + 3 pós preserva os 129 candidatos e a coorte 2009;
- o outcome tem suporte completo nas janelas candidatas (herdado do D12);
- pré-tendências em nível e em índice foram diagnosticadas, sem contaminação
  por dados pós-tratamento;
- a coorte 2009 tem uma limitação de diagnóstico reconhecida e registrada,
  não escondida.

**Ainda em aberto (não resolvido aqui, de propósito):**

- janela de antecipação;
- escolha final never-treated vs. not-yet-treated e exclusões adicionais de
  spillover;
- covariáveis de balanceamento fundamentadas por literatura;
- estratégia de inferência/clusterização;
- a especificação causal formal (ainda não congelada).

`AMOSTRA_CAUSAL_FINAL = NÃO DEFINIDA`

`DESENHO_CAUSAL_APROVADO = NÃO`

`GATE_IDENTIFICACAO = APTO_PARA_ESPECIFICACAO`

### Próxima etapa

Com o gate de identificação classificado como apto, o próximo passo é
**formalizar e congelar a especificação causal** (janela definitiva,
regras de antecipação e spillover, covariáveis de balanceamento) e só
então avaliar suporte comum/balanceamento e executar event-study/ATT.
Nenhuma dessas etapas é executada neste notebook.

## 35. Congelamento da especificação causal (D14)

O D13 classificou o gate de identificação como `APTO_PARA_ESPECIFICACAO`,
com seis ressalvas explícitas e nenhum bloqueador objetivo. O D14 tenta
transformar as **propostas** do D13 em uma **especificação causal explícita
e reproduzível** — ou identificar, com justificativa, algum ponto que ainda
impeça esse congelamento.

**O D14 não estima nenhum efeito.** Não roda Callaway–Sant'Anna, não
calcula ATT, não faz matching, não usa rede. É uma etapa inteiramente
metodológica: cada decisão abaixo segue o roteiro conceito → alternativas →
diagnóstico do projeto → decisão → implicação, e termina em uma tabela de
contrato com status explícito por item (`CONGELADO`, `SENSIBILIDADE`,
`PENDENTE` ou `BLOQUEADOR`).

**Regra fundamental que este notebook respeita:** a **janela de
event-study** (quantos períodos `k` são reportados ao redor do tratamento)
não é o mesmo que o **período total disponível para o estimador**. A
especificação principal pode reportar `k=-2,...,+2` sem descartar
observações válidas de anos mais distantes — essas continuam disponíveis
para o estimador (Callaway–Sant'Anna estima $ATT(g,t)$ para qualquer $t$ no
período total, a janela de event-study é só a forma de apresentação/
agregação).

In [20]:
import define_especificacao_causal as d14

spec = d14.ESPECIFICACAO_CAUSAL_V1
print('Especificacao D14 carregada (nenhum efeito estimado):')
for chave, valor in d14.resumo_especificacao(spec).items():
    print(f'  {chave}: {valor}')

Especificacao D14 carregada (nenhum efeito estimado):
  outcome_principal: pessoal_ocupado_assalariado
  grupo_comparacao_principal: never_treated_pool_estrutural_4964
  not_yet_treated_no_principal: False
  periodo_total: 2007-2019
  janela_principal_k: [-2, -1, 0, 1, 2]
  janela_sensibilidade_k: [-3, -2, -1, 0, 1, 2]
  k_referencia: -1
  evidencia_institucional_antecipacao: INSUFICIENTE_PARA_REGRA_GERAL
  suposicao_antecipacao_principal_periodos: 0
  painel_balanceado_exigido: True
  covariaveis_principal: []
  municipio_sigilo_excluido: 5003900
  transformacao_principal: nivel
  transformacoes_sensibilidade: ['log1p']


## 36. Definição do tratamento (recuperada, não redefinida)

**Conceito.** Presença operacional de campus associado à Expansão Fase II
no município (`CONTRATO_CAUSAL.md`). **Alternativas descartadas**: usar o
outcome CEMPRE para inferir timing (proibido — inverteria causa e efeito);
usar a data de autorização/anúncio como tratamento (não é o que a proxy do
Censo mede). **Diagnóstico do projeto**: o cadastro causal (D11) já
resolve, campo a campo, a diferença entre `ano_evento_institucional`,
`ano_transicao`, `primeiro_ano_completo` e `ano_coorte_candidata` — 128/129
candidatos sob `origem_coorte='proxy_censo'` (esses três primeiros campos
nulos) e 1/129 (Cabo Frio) sob `origem_coorte='institucional_validada'`
(todos os quatro campos preenchidos, ver seção 39). **Decisão**: o
tratamento operacional é definido exatamente como no cadastro causal
aprovado — nenhuma redefinição neste D14. **Implicação**: qualquer unidade
tratada na especificação é identificada por
`candidato_amostra_principal=True`, nunca por comportamento do CEMPRE.

## 37. Ano zero / coorte g

**Conceito.** $g$ é o primeiro ano em que a unidade está sob tratamento —
em Callaway–Sant'Anna, o "grupo" de uma unidade é o próprio ano do seu
primeiro tratamento. **Alternativas**: $g$ = primeiro ano observado no
Censo (proxy pura); $g$ = primeiro ano civil completo (quando documentado);
$g$ = ano do evento institucional (quando existe, ex.: inauguração).
**Diagnóstico**: o cadastro causal já resolve isso por município — para
128/129 (`proxy_censo`), `ano_coorte_candidata` é o primeiro ano observado;
para Cabo Frio (`institucional_validada`), `ano_coorte_candidata` já foi
preenchida com `primeiro_ano_completo=2010`, não com o ano do evento
(2009, parcial). Ou seja, **o cadastro já aplica, por unidade, a definição
"primeiro ano completo quando existe, senão a proxy"** — não são duas
regras conflitantes, é uma soma coerente de uma regra padrão com uma
exceção documentada.

**Decisão**: $g$ = `ano_coorte_candidata`, lida diretamente do cadastro
aprovado, sem exceção, para os 129. Uma única regra reproduzível cobre
todos os casos existentes — não há necessidade de forçar nem de bloquear.

In [21]:
tratados_d14 = d14.unidades_tratadas_principal(painel)
tabela_g = tratados_d14.groupby(['origem_coorte', 'ano_coorte_candidata'], as_index=False).size()
tabela_g['ano_coorte_candidata'] = tabela_g['ano_coorte_candidata'].astype(int)
tabela_g = tabela_g.rename(columns={'size': 'n_municipios'})
display(estilizar_tabela_ipt(tabela_g))
print(f"Total de unidades tratadas principais: {len(tratados_d14)} (esperado 129).")
print(f"Regra de g cobre 100% dos casos: {tabela_g['n_municipios'].sum() == len(tratados_d14)}")

,origem_coorte,ano_coorte_candidata,n_municipios
0,institucional_validada,2010,1
1,proxy_censo,2009,21
2,proxy_censo,2010,26
3,proxy_censo,2011,66
4,proxy_censo,2012,13
5,proxy_censo,2013,2


Total de unidades tratadas principais: 129 (esperado 129).
Regra de g cobre 100% dos casos: True


## 38. Antecipação: evidência institucional versus suposição da especificação

**Conceito.** Antecipação é o efeito que o tratamento pode produzir sobre o
outcome **antes** do ano g observado — por exemplo, se a expectativa de
um campus já eleva a atividade local por antecipação de obras/contratações
preparatórias. Callaway–Sant'Anna trata isso como um parâmetro explícito
(quantos períodos antes de g já podem estar "contaminados").

**Duas ideias que este notebook separa deliberadamente, para não serem
confundidas:**

- **A. Evidência institucional sobre antecipação** — o que os documentos
  aprovados (`CONTRATO_CAUSAL.md`, `PROTOCOLO_PRE_ANALISE.md`, seção 6.7)
  permitem afirmar empiricamente sobre uma janela de antecipação geral;
- **B. Suposição usada na especificação causal** — o parâmetro que a
  especificação principal precisa fixar para poder identificar o efeito,
  mesmo sem evidência definitiva.

Não registrar simplesmente `JANELA_ANTECIPACAO_DEFINIDA = NÃO` como se
nenhuma decisão operacional existisse — a especificação PRECISA de um
valor para o parâmetro de antecipação do estimador, e um valor foi de fato
escolhido (B), mesmo sem evidência (A) que o comprove.

**Alternativas para B (do enunciado do D14):**

- A. nenhuma antecipação (parâmetro = 0, o padrão da literatura/pacote);
- B. 1 período de antecipação;
- C. regra baseada nos campos institucionais existentes;
- D. outra solução suportada pelo contrato.

**Diagnóstico do projeto.** `CONTRATO_CAUSAL.md` e `PROTOCOLO_PRE_ANALISE.md`
(seção 6.7) são explícitos: não existe, hoje, uma janela de antecipação
geral decidida — nem 0, nem 1, nem qualquer outra, para os 128/129 sob
`proxy_censo`. O único caso com regra institucional (`ano_transicao`
documentado) é Cabo Frio — e ali a regra resolve uma questão diferente
(qual ano é "parcial" dentro do próprio tratamento, não um efeito antes de
g; ver seção 39). Não há, portanto, evidência para C como regra geral.

**Registro explícito das duas dimensões:**

`EVIDENCIA_INSTITUCIONAL_ANTECIPACAO = INSUFICIENTE_PARA_REGRA_GERAL` —
não foi encontrada evidência institucional geral que permita afirmar
empiricamente ausência de antecipação para os 128 candidatos sob proxy do
Censo. Isso é uma **limitação**, não uma decisão.

`SUPOSICAO_ANTECIPACAO_PRINCIPAL = 0_PERIODOS` — a especificação
principal adota **0 períodos como hipótese identificadora**, não como fato
observado. É a opção mais conservadora disponível sem inventar uma janela
sem evidência, e coincide com o parâmetro padrão (`anticipation=0`) do
Callaway–Sant'Anna.

**Cabo Frio permanece tratado pela regra institucional específica já
existente**, distinta desta suposição geral: 2009 = transição, excluída da
estimação; 2010 = primeiro ano completo = g = k=0 (seção 39).

**Implicação / limitação registrada, não sensibilidade nova nesta etapa:**
os diagnósticos de pré-tendência já feitos no D13 (níveis, índice, mudança
g-2 → g-1) são o instrumento qualitativo disponível para avaliar essa
suposição — e não mostraram sinal que a contradiga de forma flagrante.
Uma análise de sensibilidade formal com antecipação ≠ 0 fica registrada
como limitação/robustez para etapa futura, não implementada aqui.

## 39. Cabo Frio/RJ — regra do estimador futuro

**Não recalculado aqui** — a regra já existe no cadastro causal aprovado
(D11), este D14 apenas a formaliza como regra do estimador futuro:

- **2009** = `ano_transicao`, já presente em `anos_excluir_estimacao` no
  cadastro causal → **não deve ser usado como período pré nem como
  período pós** na construção do event-study para este município: é um
  ano parcial, explicitamente excluído da estimação;
- **2010** = `primeiro_ano_completo` = `ano_coorte_candidata` → **é `k=0`**
  para Cabo Frio, exatamente como para qualquer outro tratado;
- os anos 2007-2008 (pré) e 2011-2013 (pós, dentro da janela principal)
  entram normalmente.

**Decisão**: ao construir `event_time` para Cabo Frio, os anos em
`anos_excluir_estimacao` (aqui, só 2009) são removidos do painel usado pelo
estimador para esse município especificamente — os demais 128 têm essa
lista vazia e não são afetados. Isso é uma regra por unidade, já suportada
pelo cadastro, sem necessidade de exceção ad-hoc no código do estimador.

In [22]:
anos_excluidos = d14.anos_excluidos_por_municipio(painel)
print(f"Cabo Frio ({d14.CODIGO_CABO_FRIO}) -- anos excluidos da estimacao: {anos_excluidos[d14.CODIGO_CABO_FRIO]}")
n_com_excluir = sum(1 for v in anos_excluidos.values() if v)
print(f"Municipios com algum ano excluido: {n_com_excluir} de {len(anos_excluidos)} (esperado 1 de 129).")

Cabo Frio (3300704) -- anos excluidos da estimacao: [2009]
Municipios com algum ano excluido: 1 de 129 (esperado 1 de 129).


## 40. Grupo de comparação principal: never-treated

**Conceito.** Never-treated são unidades que nunca observam o tratamento em
toda a janela do painel (2007-2019) — candidatas "limpas" de comparação
porque seu outcome nunca é mecanicamente afetado pelo próprio tratamento
que estamos medindo. **Alternativa** (not-yet-treated) é avaliada
separadamente na seção 41.

**Vantagens do never-treated aqui:** (1) o pool de 4.964 já foi auditado
(D13): subconjunto estrito dos 4.970 nunca expostos, nenhum Fase II —
zero contaminação por tratamento observado; (2) grande (4.964, ou 4.963
após excluir o sigilo — seção 48), preservando poder estatístico; (3) não
exige nenhuma regra adicional de corte temporal (todo o painel 2007-2019
serve de pré e pós, sem risco de "usar unidade já tratada como controle").

**Limitações:** os 4.964 não são never-treated *validados
institucionalmente* um a um (ao contrário dos 147 Fase II) — são "sem
exposição observada" pela mesma proxy usada para o tratamento (mesmo tipo
de limitação de medição, não uma falha adicional); podem incluir
municípios afetados por outras políticas federais/estaduais concomitantes
não capturadas pelo cadastro nacional de exposição (risco genérico de
qualquer grupo de comparação, mitigado por Callaway–Sant'Anna controlar
tendência comum, não eliminado por completo).

**Decisão**: **never-treated do pool estrutural de 4.964 (4.963 após
excluir o sigilo) como grupo de comparação principal.** Nenhum matching,
nenhum filtro por outcome ou trajetória.

In [23]:
controles_d14 = d14.unidades_controle_principal(painel)
print(f"Controles principais (never-treated, pool 4964 menos sigilo): {len(controles_d14)}")
print(f"Esperado: 4963. Confere: {len(controles_d14) == 4963}")

Controles principais (never-treated, pool 4964 menos sigilo): 4963
Esperado: 4963. Confere: True


## 41. Not-yet-treated: avaliado, não implementado

**Conceito.** Not-yet-treated usaria, no ano $t$, os municípios Fase II que
**ainda não** foram tratados naquele ano como comparação temporária para os
que já foram — descartados como comparação assim que eles próprios são
tratados.

**Quem seriam, em cada $t$, entre os 129?** A tabela abaixo mostra: por
exemplo, em 2007-2008 todos os 129 ainda não tratados; em 2011 já são 114
tratados e restam só 15 not-yet-treated; a partir de 2013 não sobra nenhum
(os cinco anos finais do painel, 2014-2019, não têm nenhuma unidade
not-yet-treated disponível).

In [24]:
not_yet = []
for t in range(2007, 2020):
    ja_tratados = int((tratados_d14['ano_coorte_candidata'] <= t).sum())
    ainda_nao = int((tratados_d14['ano_coorte_candidata'] > t).sum())
    not_yet.append({'ano': t, 'ja_tratados': ja_tratados, 'not_yet_treated': ainda_nao})
tabela_not_yet = pd.DataFrame(not_yet)
display(estilizar_tabela_ipt(tabela_not_yet))

,ano,ja_tratados,not_yet_treated
0,2007,0,129
1,2008,0,129
2,2009,21,108
3,2010,48,81
4,2011,114,15
5,2012,127,2
6,2013,129,0
7,2014,129,0
8,2015,129,0
9,2016,129,0


**Distinção conceitual que precisa ficar explícita.** `pode_ser_controle=False`
no cadastro causal significa que os 147 municípios Fase II não pertencem
ao **pool estrutural de controles permanentes** (a lista de candidatos a
controle construída em `POOL_CANDIDATO_CONTROLES.md`, sem exposição
observada em 2007-2019). Isso é um conceito diferente de usar unidades
**ainda não tratadas** como grupo de comparação econométrico antes de sua
própria adoção (not-yet-treated) — Callaway–Sant'Anna pode, conceitualmente,
trabalhar com grupos never-treated ou not-yet-treated dependendo da
especificação escolhida; a flag `pode_ser_controle` não é, por si só, uma
proibição técnica desse desenho.

Neste projeto, escolhemos **deliberadamente** never-treated como grupo de
comparação principal — não porque not-yet-treated seja metodologicamente
inválido em geral, mas pelas razões abaixo:

1. **Suficiência operacional**: existem 4.963 controles never-treated
   válidos (seção 40) — não há necessidade operacional de ampliar o
   conjunto de comparação incorporando not-yet-treated;
2. **Simplicidade de interpretação**: o contrafactual "nunca foi exposto
   em 2007-2019" é mais simples de interpretar que um contrafactual que
   muda de papel (comparação → tratado) ao longo do painel;
3. **Incerteza de timing**: o timing dos futuros tratados (não-2009,
   ainda não tratados em um dado $t$) carrega a mesma incerteza
   institucional/proxy do Censo que qualquer outro candidato (seção 36) —
   usá-los como comparação herdaria essa incerteza para o lado do
   contrafactual, não só do tratamento;
4. **Antecipação não é empiricamente conhecida**: a seção 38 registra que
   não há evidência institucional geral sobre antecipação — se ela
   existir e not-yet-treated for usado como comparação, o "controle"
   (ainda não tratado por definição de $g$) poderia já estar antecipando
   efeitos, contaminando exatamente a comparação que deveria ser limpa;
5. **Risco de contaminação por mudança de comportamento pré-tratamento**:
   usar not-yet-treated introduz risco adicional se o comportamento da
   unidade mudar antes do ano de tratamento registrado — risco que
   never-treated, por definição, não carrega.

**Isso agrega informação relevante dado que já temos 4.963 never-treated?**
Não de forma decisiva: not-yet-treated é pequeno e encolhe rápido (81 em
2010, 15 em 2011, 2 em 2012, 0 a partir de 2013 — tabela acima), exatamente
nos anos em que mais precisaríamos dele (coortes tardias). Comparado aos
4.963 never-treated, o ganho de informação é marginal e concentrado nas
coortes com mais risco de antecipação — exatamente o cenário em que
not-yet-treated é mais arriscado (razão 4 acima).

**Decisão: A — não entra no desenho** (nem principal, nem sensibilidade,
nesta especificação), por escolha metodológica deliberada e fundamentada
nas cinco razões acima — não por proibição técnica do cadastro. Não é uma
rejeição permanente: se a antecipação for investigada e descartada com
evidência futura, not-yet-treated poderia voltar a ser avaliado como
sensibilidade — mas isso seria uma nova decisão metodológica formal, não
uma reabertura silenciosa deste D14.

`GRUPO_COMPARACAO_PRINCIPAL = NEVER_TREATED`

`NOT_YET_TREATED = NÃO UTILIZADO`

## 42. Tendências paralelas: incondicional ou condicional a X?

O D13 encontrou diferença importante de **nível** do CEMPRE 708 entre
tratados e controles no baseline (mediana dos tratados ~14x maior que a dos
controles). Isso não reprova DiD automaticamente — tendências paralelas
não exige nível igual, exige trajetória comparável (D13, seção 20).

**Duas hipóteses possíveis:**

- **Incondicional**: assume que, na ausência do tratamento, tratados e
  controles teriam trajetórias paralelas *sem* precisar controlar por
  nenhuma covariável;
- **Condicional a X**: assume paralelismo só *depois* de condicionar em
  covariáveis pré-tratamento (ex.: população, PIB per capita, composição
  setorial) que capturam por que o MEC selecionou aqueles municípios.

O DAG do `CONTRATO_CAUSAL.md` (características municipais prévias → seleção
→ tratamento, e características prévias → outcome) é exatamente a
justificativa teórica para preferir a versão condicional **se** covariáveis
adequadas estiverem disponíveis. A seção 44 audita se estão.

## 43. DAG: confundidor, mediador, outcome, tratamento

Recuperando o DAG já aprovado em `CONTRATO_CAUSAL.md` (não uma nova
análise):

- **Confundidor pré-tratamento (X)** — características municipais prévias
  (pobreza, população, PIB, setor produtivo, infraestrutura). Afetam tanto
  a seleção do MEC (`S`) quanto o outcome (`Y`), independentemente do
  tratamento. São as únicas variáveis legítimas para ajuste, **se**
  disponíveis e medidas antes do tratamento.
- **Seleção/implantação (S)** — elo entre X e o tratamento; não é uma
  variável a controlar, é o mecanismo que explica por que X confunde.
- **Tratamento (P)** — presença operacional do campus (seção 36).
- **Mediadores pós-tratamento (M)** — contratação direta, consumo local,
  atração de empresas, formação de capital humano. **Consequência** do
  tratamento — nunca devem ser usados como covariável de ajuste (isso
  bloquearia parte do próprio efeito que queremos medir).
- **Choques concomitantes (Z)** — ciclo econômico nacional, outras
  políticas — afetam `Y` diretamente, sem relação com `P`; justificam
  inferência robusta a tendências comuns (Callaway–Sant'Anna), não uma
  covariável de ajuste.
- **Outcome (Y)** — CEMPRE 708.

**Classificação das variáveis do DAG frente aos dados hoje disponíveis:**

| Variável do DAG | Já existe nos dados do projeto? | Pode ser usada como covariável de ajuste? |
|---|---|---|
| População municipal | Não (busca confirmada abaixo) | Seria candidata legítima (X) — mas indisponível |
| PIB/renda municipal | Não | Seria candidata legítima (X) — mas indisponível |
| Composição setorial prévia | Parcialmente no próprio CEMPRE (pessoal ocupado total, unidades locais) — não auditado como covariável nesta etapa | Candidata condicionalmente legítima, requer auditoria própria antes de usar |
| Infraestrutura prévia | Não | Seria candidata legítima (X) — mas indisponível |
| Contratação direta, consumo local, atração de empresas (mediadores) | Não relevante — são mediadores | **Nunca** deve ser usada como covariável de ajuste |
| Ciclo econômico nacional / outras políticas (Z) | Não modelado como covariável | Não é covariável de X; é justificativa para o desenho de inferência |

O DAG é hipótese de trabalho, não prova — a revisão de literatura para
validar quais covariáveis de X realmente pertencem à especificação segue
pendente (`CONTRATO_CAUSAL.md`, nota final do DAG; `PROTOCOLO_PRE_ANALISE.md`,
seção 6.9).

## 44. Gate de covariáveis

**Diagnóstico**: nenhuma variável de X (população, PIB, composição
setorial validada, infraestrutura) está hoje disponível como artefato
aprovado em `data/processed/` — confirmado por busca direta abaixo, sem
abrir nova frente de coleta nesta tarefa. As únicas variáveis
pré-tratamento no projeto são as do próprio CEMPRE (outros indicadores de
emprego/estabelecimentos), que não foram auditadas nem aprovadas como
covariáveis de balanceamento.

`COVARIAVEIS_ESPECIFICACAO = D — SENSIBILIDADE, NÃO PRINCIPAL`

**Por quê não A (não escolhida por economia de esforço):** o DAG dá uma
razão causal concreta para preferir condicionar em X (a seleção do MEC não
é aleatória, e a diferença de nível do D13 é consistente com essa
hipótese) — ignorar isso sem registrar seria escolher A só por
conveniência, o que o D14 proíbe explicitamente.

**Por quê não C (não escolhida sem risco causal concreto adicional):**
"necessárias mas faltantes" implicaria que a especificação principal *não
pode* ser congelada sem covariáveis — mas Callaway–Sant'Anna aceita
tendências paralelas incondicionais como identificação válida (mais forte,
não inválida), e o próprio D13 já registrou a diferença de nível como
diagnóstico, não como bloqueador. Bloquear o congelamento inteiro por
covariáveis ausentes seria desproporcional ao risco hoje evidenciado.

**Decisão**: a especificação **principal** assume tendências paralelas
**incondicionais** (sem covariáveis) — suposição mais forte, explicitada
como tal, não provada. Covariáveis de X (quando alguma fonte populacional/
socioeconômica for aprovada no projeto) ficam registradas como **análise de
sensibilidade futura** (condicional), não como pré-requisito do
congelamento atual.

In [25]:
import glob
candidatos_covariaveis = [c for c in ('populacao', 'pib', 'renda', 'idh') if glob.glob(str(ROOT / 'data' / 'processed' / f'*{c}*'))]
print(f"Arquivos de covariaveis socioeconomicas encontrados em data/processed/: {candidatos_covariaveis or 'nenhum'}")
print(f"COVARIAVEIS_ESPECIFICACAO = D (SENSIBILIDADE_NAO_PRINCIPAL)")
print(f"covariaveis_principal (config congelada): {list(spec.covariaveis_principal)}")

Arquivos de covariaveis socioeconomicas encontrados em data/processed/: nenhum
COVARIAVEIS_ESPECIFICACAO = D (SENSIBILIDADE_NAO_PRINCIPAL)
covariaveis_principal (config congelada): []


## 45. Outcome principal

**Candidato**: CEMPRE 708 — pessoal ocupado assalariado, em nível.

**Vantagem**: interpretação direta em unidades reais — número de pessoas
com emprego formal assalariado. Qualquer efeito estimado tem leitura
imediata ("X empregos a mais/a menos"), sem exigir explicar uma
elasticidade ou índice abstrato.

**Limitação**: heterogeneidade de escala municipal é grande (seção 46) —
municípios variam de algumas dezenas a centenas de milhares de
empregos — o que pode tornar a média simples sensível a poucos municípios
grandes. Essa é exatamente a limitação que a auditoria de transformação
(seção 46) e o outcome per capita (seção 47) avaliam.

## 46. Transformação do outcome: auditoria de escala

Distribuição do CEMPRE 708 em todo o período 2007-2019, separando tratados
(129 candidatos) e o pool de controles principal (4.963, após excluir o
sigilo).

In [26]:
import numpy as np

tratados_full = painel.loc[painel['candidato_amostra_principal'] == True, 'pessoal_ocupado_assalariado'].dropna()
controles_full = painel.loc[
    (painel['fl_elegivel_controle_candidato'] == True) & (painel['codigo_municipio_ibge'] != d14.CODIGO_MUNICIPIO_SIGILO_EXCLUIR),
    'pessoal_ocupado_assalariado',
].dropna()

auditoria_transformacao = pd.DataFrame([
    {
        'grupo': nome, 'n': len(serie), 'minimo': serie.min(), 'p25': serie.quantile(.25),
        'mediana': serie.median(), 'p75': serie.quantile(.75), 'maximo': serie.max(),
        'n_zeros': int((serie == 0).sum()),
    }
    for nome, serie in [('tratados', tratados_full), ('controles', controles_full)]
])
display(estilizar_tabela_ipt(auditoria_transformacao))

fig = make_subplots(rows=1, cols=2, subplot_titles=['Nivel', 'log1p'])
for nome, serie, cor in [('tratados', tratados_full, CORES_IPT['AZUL_PRINCIPAL']), ('controles', controles_full, CORES_IPT['CIANO'])]:
    fig.add_trace(go.Histogram(x=serie, name=nome, marker_color=cor, opacity=0.65, showlegend=True), row=1, col=1)
    fig.add_trace(go.Histogram(x=np.log1p(serie), name=nome, marker_color=cor, opacity=0.65, showlegend=False), row=1, col=2)
aplicar_tema_ipt(fig, titulo='Distribuicao do CEMPRE 708 (2007-2019): nivel vs log1p')
fig.update_layout(barmode='overlay')
fig.update_xaxes(title_text='Pessoal ocupado assalariado', row=1, col=1)
fig.update_xaxes(title_text='log1p(Pessoal ocupado assalariado)', row=1, col=2)
salvar_figura_ipt(fig, FIGURES / '10_distribuicao_outcome_nivel_log1p.png')
fig.show()

,grupo,n,minimo,p25,mediana,p75,maximo,n_zeros
0,tratados,1677,176.000000,4419.000000,11024.000000,24747.000000,179317.000000,0
1,controles,64519,0.000000,391.000000,783.000000,1942.000000,292385.000000,3


**Diagnóstico**: 3 zeros reais existem no pool de controles (nenhum nos
tratados) — `log()` puro seria indefinido para essas células; `log1p()` é a
opção segura se uma transformação logarítmica for usada. A distribuição em
nível é fortemente assimétrica à direita nos dois grupos (mediana muito
abaixo da média, cauda longa) — típico de dados de contagem econômica
municipal.

**Decisão**:

- **A — nível como principal** (`transformacao_principal='nivel'`):
  mantém a interpretação direta em empregos (seção 45) e é o outcome já
  formalizado no `CONTRATO_CAUSAL.md`, decidido antes de qualquer
  observação de efeito — não deve ser trocado agora só porque a
  distribuição é assimétrica;
- **C — log1p como sensibilidade** (não B — não existe registro
  metodológico prévio validando log1p como principal, e a decisão do
  contrato já é nível): registrado como transformação de sensibilidade
  exatamente pela presença de zeros reais nos controles.

Nenhuma transformação foi usada para "melhorar" pré-tendência nem para
escolher entre nível/log com base em como o gráfico ficou mais bonito.

## 47. Outcome per capita: não disponível

Verificado diretamente no projeto (sem buscar dado novo): nenhuma coluna de
população municipal existe no painel integrado, e nenhum arquivo de
população aprovado existe em `data/processed/` (busca da seção 44 já
cobriu isso — `populacao` não aparece entre os artefatos existentes).

**Decisão**: outcome per capita **não está disponível nesta especificação**.
Não é aberta nenhuma nova frente de coleta de dado populacional nesta
tarefa — isso ficaria registrado como possível trabalho futuro, condicional
a uma fonte populacional municipal ser aprovada em etapa própria, com sua
própria auditoria de proveniência (como já é o padrão do projeto).

## 48. Sigilo do município 5003900 em 2012

**Fato**: uma única célula do pool estrutural (4.964) tem CEMPRE 708 =
sigilo em 2012 — código `5003900`. Confirmado abaixo, direto do painel,
sem alteração.

**Alternativas avaliadas** (proibido: imputar valor; proibido: substituir
por zero):

- excluir o município do painel causal principal inteiro (todos os anos,
  não só 2012), preservando painel balanceado;
- manter o município com um buraco em 2012 e usar um estimador que aceite
  painel não balanceado (Callaway–Sant'Anna suporta isso tecnicamente, mas
  exige decisão explícita de implementação futura).

**Decisão**: **excluir o município `5003900` do painel causal principal
inteiro** — não é uma unidade tratada, é 1 controle em 4.964 (0,02%), a
perda de informação é desprezível e a exclusão evita introduzir a
complexidade adicional de um painel não balanceado só por causa de uma
única célula. `municipio_sigilo_excluido = '5003900'`.

In [27]:
linha_sigilo = painel.loc[(painel['codigo_municipio_ibge'] == d14.CODIGO_MUNICIPIO_SIGILO_EXCLUIR) & (painel['ano'] == 2012)]
display(estilizar_tabela_ipt(linha_sigilo[['codigo_municipio_ibge', 'ano', 'pessoal_ocupado_assalariado', 'status_pessoal_ocupado_assalariado', 'fl_elegivel_controle_candidato']]))
print(f"Impacto: 1 municipio em {int(painel.loc[painel['fl_elegivel_controle_candidato'] == True, 'codigo_municipio_ibge'].nunique())} controles estruturais ({1/4964:.4%}).")

,codigo_municipio_ibge,ano,pessoal_ocupado_assalariado,status_pessoal_ocupado_assalariado,fl_elegivel_controle_candidato
66760,5003900,2012,nan,sigilo,True


Impacto: 1 municipio em 4964 controles estruturais (0.0201%).


## 49. Painel balanceado

Com a exclusão do município `5003900` (seção 48), o painel causal principal
fica **naturalmente balanceado**: os 129 tratados têm 1.677/1.677
observações utilizáveis de 708 (D12), e os 4.963 controles restantes não
têm nenhuma célula sigilosa/indisponível/missing dentro de 2007-2019 (D13,
auditoria do pool). Não é necessário decidir entre "aceitar buraco" e
"excluir" para mais nenhum caso — o único caso existente já foi resolvido
na seção 48.

**Decisão**: `painel_balanceado_exigido = True` na especificação
principal. Isso não é uma exigência técnica do Callaway–Sant'Anna (que
aceita desbalanceamento), é uma escolha de simplicidade desta
especificação, possível porque o dado hoje já permite um painel
balanceado sem perda relevante de informação (0,02%). Se uma etapa futura
de implementação do estimador precisar reavaliar isso, fica registrado
aqui como dependência explícita, não decidida silenciosamente no código do
estimador.

## 50. Período total da estimação

**Não confundir com a janela de event-study (seção 51-52).** O painel
disponível é 2007-2019 inteiro. Não há, nos documentos aprovados, nenhuma
justificativa causal para cortar anos do período total — cortar reduziria
informação válida sem ganho de identificação.

**Decisão**: a estimação principal futura usará **todo o período
2007-2019** disponível no painel — `periodo_total_inicio=2007`,
`periodo_total_fim=2019`. A janela de event-study (k=-2..+2 ou k=-3..+2)
é apenas a forma de **apresentação/agregação** dos resultados por tempo
relativo ao tratamento — Callaway–Sant'Anna pode estimar $ATT(g,t)$ para
qualquer $t$ dentro do período total, mesmo além da janela reportada no
event-study principal.

## 51. Janela principal de event-study

**Candidata**: `k = -2, -1, 0, +1, +2` (equivalente a "2 pré + 3 pós",
contando $g$ como o primeiro período tratado, $k=0$).

**Diagnóstico**: o D13 já comparou esta janela com a de 3 pré + 3 pós e
mostrou 129/129 candidatos preservados contra 108/129 (D13, seção 16-17) —
sem bloqueador identificado nas seções deste D14 que a contradiga.

**Confirmação**: **k = -2, -1, 0, +1, +2 confirmada como janela principal**
de apresentação/event-study.

**Período de referência**: `k = -1` é fixado como período de referência
futuro (o período imediatamente anterior ao tratamento, contra o qual os
demais $k$ serão comparados na normalização do event-study) — consistente
com o índice `g-1=100` já usado como visualização descritiva no D13.

## 52. Janela de sensibilidade

**Candidata**: `k = -3, -2, -1, 0, +1, +2` ("3 pré + 3 pós").

**Confirmação**: mantida como sensibilidade, não principal — exclui a
coorte 2009 inteira por construção (precisaria de 2006, fora do painel).
Participam apenas as coortes 2010-2013.

In [28]:
tabela_gate_d14 = d13.tabela_gate_por_coorte(painel)
sensibilidade_coortes = tabela_gate_d14.loc[tabela_gate_d14['suporte_3pre3pos'] > 0]
print(f"Coortes na janela de sensibilidade (3pre+3pos): {sensibilidade_coortes['ano_coorte_candidata'].astype(int).tolist()}")
print(f"Tratados na sensibilidade: {int(sensibilidade_coortes['suporte_3pre3pos'].sum())} de 129")

Coortes na janela de sensibilidade (3pre+3pos): [2010, 2011, 2012, 2013]
Tratados na sensibilidade: 108 de 129


## 53. Coorte 2009: decisão final

**Principal**: **incluída**. Sob a janela principal (2 pré + 3 pós), a
coorte 2009 tem suporte completo (21/21, D12/D13) — não há razão para
excluí-la só porque tem menos pré-períodos disponíveis (regra explícita do
D14: "não excluí-la apenas porque possui menos pré-períodos").

**Sensibilidade (3 pré)**: **excluída por construção**, não por análise
separada — o painel simplesmente não alcança 2006. Isso é uma limitação de
calendário, documentada, não uma escolha metodológica arbitrária.

**Limitação registrada**: a coorte 2009 tem só uma mudança pré observável
(2007→2008, D13 seção 26-27) — a capacidade de diagnosticar pré-tendência
para essa coorte específica é menor que para as demais, mesmo estando
incluída na especificação principal.

## 54. Spillover: política futura

Diagnósticos já existentes (não reabertos, não recalculados nesta etapa):
`DIAGNOSTICO_SPILLOVER_FASE_II.md` (distância sede-a-sede, limiares
exploratórios de 25/50/100 km) e `DIAGNOSTICO_ARRANJOS_POPULACIONAIS_FASE_II.md`
(compartilhamento de arranjo populacional com município Fase II). Ambos
são explicitamente diagnósticos, não regras de exclusão automática
(`CONTRATO_CAUSAL.md`, seção "Spillovers"; D13, seção 30).

**Alternativas**: A. nenhum filtro no principal, com sensibilidade
espacial disponível; B. filtro pré-definido já no principal; C. ainda
requer decisão.

**Decisão: A** — nenhum filtro geográfico na especificação **principal**
(usa o pool de controles inteiro, seção 40). Nenhum raio de exclusão tem
justificativa substantiva documentada (dados de deslocamento pendular) que
permita escolher um valor específico sem arbitrariedade — e o D14 proíbe
inventar novos quilômetros. Os limiares **já documentados** no projeto
(25/50/100 km de distância sede-a-sede; flag de arranjo populacional
compartilhado) ficam registrados como **candidatos a sensibilidade
espacial futura** — não escolhidos aqui como *o* critério definitivo,
apenas listados como o que já existe pronto para uso, se e quando uma
análise de robustez espacial for decidida.

## 55. Estimando (estimand) principal candidato

Refinado a partir das decisões acima, em linguagem natural — **nenhum
valor é apresentado, isto não é um resultado**:

> O efeito médio da presença operacional de um campus da Expansão Fase II
> sobre o pessoal ocupado assalariado (CEMPRE 708) dos 129 municípios
> candidatos principais, em cada coorte de tratamento $g$ e cada ano
> calendário $t$ do período 2007-2019, comparado ao contrafactual
> representado pelo grupo never-treated do pool estrutural de 4.963
> controles elegíveis — sob as suposições de tendências paralelas
> incondicionais e ausência de antecipação — reportado principalmente na
> janela de event-time $k \in \{-2,...,+2\}$ com $k=-1$ como referência, e
> agregável por coorte e por tempo relativo ao tratamento.

## O que vamos estimar: ATT(g,t)

Em Callaway–Sant'Anna, o efeito não é um único número — é uma função de
duas coordenadas:

- $g$ = coorte de tratamento (o ano em que o município passou a ter campus
  operacional);
- $t$ = ano calendário em que o outcome é observado.

$$ATT(g,t) = E\big[Y_t(g) - Y_t(\infty) \mid G=g\big], \quad t \ge g$$

**Exemplo simples (sem calcular resultado real):** para $g=2011$ e $t=2012$,
$ATT(2011, 2012)$ seria o efeito médio, no ano 2012, sobre os municípios
que foram tratados pela primeira vez em 2011 — um ano depois do
tratamento começar para essa coorte especificamente ($k=+1$).

**Por que isso é útil com tratamento escalonado**: em vez de forçar um
único coeficiente médio (como TWFE ingênuo faria, seção 59), $ATT(g,t)$
mantém cada coorte separada até o momento de agregar — nenhuma coorte
"contamina" a estimativa de outra coorte antes da agregação ser uma
escolha explícita, não um acidente da regressão.

## Event time

$$k = t - g$$

**Exemplo (g=2011):**

| t | k |
|---|---|
| 2009 | -2 |
| 2010 | -1 |
| 2011 | 0 |
| 2012 | +1 |

Calculado abaixo apenas como exemplo didático em memória — **nenhuma
coluna `event_time` é criada em nenhum artefato persistido**, porque a
especificação, embora congelada nesta etapa, ainda não foi usada para
construir a amostra causal materializada (seção 62).

In [29]:
exemplo_g = 2011
for t in [2009, 2010, 2011, 2012]:
    print(f"g={exemplo_g}, t={t} -> k={d14.event_time(t, exemplo_g)}")

g=2011, t=2009 -> k=-2
g=2011, t=2010 -> k=-1
g=2011, t=2011 -> k=0
g=2011, t=2012 -> k=1


### Por que Callaway–Sant'Anna é candidato para este problema

- **Tratamento escalonado**: estima $ATT(g,t)$ separadamente por coorte
  $g$ e ano $t$ (seção 56), em vez de um único coeficiente médio — evita o
  problema de comparações implícitas entre unidades já tratadas e
  recém-tratadas que contamina TWFE ingênuo sob adoção escalonada (D13,
  seção 19);
- **Grupos de comparação válidos**: por padrão, compara cada coorte $g$
  contra never-treated (ou not-yet-treated, se autorizado — aqui, não,
  seção 41) — nunca contra outra coorte já tratada;
- **Agregação posterior explícita**: depois de estimar $ATT(g,t)$ para
  cada célula, a agregação (por coorte, por tempo relativo ao tratamento,
  ou um único número geral) é uma escolha separada e documentável, não
  embutida silenciosamente na regressão.

**O que o método não resolve sozinho** — precisa das decisões deste D14,
não as substitui: timing errado (depende do cadastro causal, seção 36-37);
antecipação (parâmetro que precisa ser informado, seção 38); spillover
(contaminação do grupo de comparação, seção 54); confundimento
(tendências paralelas incondicional vs. condicional, seção 42-44);
tendências paralelas implausíveis (diagnóstico, não garantia — D13).
**Nenhum estimador foi executado nesta etapa.**

### Por que TWFE ingênuo não será o estimador principal

Já registrado no D13 (seção 19) e no `CONTRATO_CAUSAL.md`: com adoção
escalonada e efeitos heterogêneos por coorte/tempo, uma regressão de
efeitos fixos de município e ano com uma única variável `tratado × pós`
pode implicitamente comparar unidades já tratadas contra unidades
recém-tratadas — misturando o efeito verdadeiro com diferenças espúrias
entre coortes. Sem entrar na derivação matemática: esse é o motivo
suficiente para não usar TWFE ingênuo como estimador causal principal
aqui. **Nenhuma regressão TWFE foi executada nesta etapa.**

## 60. Tabela — contrato da especificação

In [30]:
contrato_especificacao = pd.DataFrame([
    {'Item': 'Tratamento', 'Decisao': 'Presenca operacional de campus Fase II (cadastro causal D11, nao redefinido)', 'Status': 'CONGELADO', 'Justificativa': 'Definicao ja aprovada; secao 36'},
    {'Item': 'Coorte g', 'Decisao': "g = ano_coorte_candidata (campo canonico do cadastro)", 'Status': 'CONGELADO', 'Justificativa': 'Regra unica cobre os 129; secao 37'},
    {'Item': 'Ano zero', 'Decisao': 'k=0 no ano g; para Cabo Frio, g=2010 (primeiro ano completo)', 'Status': 'CONGELADO', 'Justificativa': 'Secao 37 e 39'},
    {'Item': 'Evidencia institucional de antecipacao', 'Decisao': 'Insuficiente para regra geral (128/129 sob proxy_censo, sem transicao documentada)', 'Status': 'LIMITACAO', 'Justificativa': 'CONTRATO_CAUSAL.md/PROTOCOLO_PRE_ANALISE.md 6.7; secao 38'},
    {'Item': 'Suposicao de antecipacao principal', 'Decisao': '0 periodos (hipotese identificadora, nao fato observado)', 'Status': 'CONGELADO', 'Justificativa': 'Padrao Callaway-Sant Anna; secao 38'},
    {'Item': 'Cabo Frio', 'Decisao': '2009 = transicao, excluida da estimacao; 2010 = primeiro ano completo = g = k=0', 'Status': 'CONGELADO', 'Justificativa': 'Regra ja no cadastro causal; secao 39'},
    {'Item': 'Tratados principais', 'Decisao': '129 candidatos (candidato_amostra_principal=True)', 'Status': 'CONGELADO', 'Justificativa': 'D12/D13/D14, reproduzido; secao 37'},
    {'Item': 'Controle principal', 'Decisao': 'Never-treated, pool estrutural 4.964 menos sigilo = 4.963', 'Status': 'CONGELADO', 'Justificativa': 'Secao 40 e 48'},
    {'Item': 'Not-yet-treated', 'Decisao': 'Nao utilizado (escolha metodologica deliberada, nao proibicao tecnica)', 'Status': 'CONGELADO', 'Justificativa': 'Cinco razoes explicitas; secao 41'},
    {'Item': 'Periodo total', 'Decisao': '2007-2019 (painel inteiro)', 'Status': 'CONGELADO', 'Justificativa': 'Nenhuma justificativa causal para cortar; secao 50'},
    {'Item': 'Janela principal', 'Decisao': 'k = -2,-1,0,+1,+2 (2 pre + 3 pos)', 'Status': 'CONGELADO', 'Justificativa': '129/129 preservados; secao 51'},
    {'Item': 'Janela sensibilidade', 'Decisao': 'k = -3,-2,-1,0,+1,+2 (3 pre + 3 pos)', 'Status': 'SENSIBILIDADE', 'Justificativa': '108/129, exclui 2009; secao 52'},
    {'Item': 'Referencia futura', 'Decisao': 'k=-1 como periodo de referencia do event-study', 'Status': 'CONGELADO', 'Justificativa': 'Secao 51'},
    {'Item': 'Coorte 2009', 'Decisao': 'Incluida no principal; excluida por construcao na sensibilidade 3pre', 'Status': 'CONGELADO', 'Justificativa': 'Secao 53'},
    {'Item': 'Outcome', 'Decisao': 'CEMPRE 708 (pessoal ocupado assalariado), nivel', 'Status': 'CONGELADO', 'Justificativa': 'CONTRATO_CAUSAL.md; secao 45'},
    {'Item': 'Transformacao', 'Decisao': 'log1p como sensibilidade (ha zeros reais nos controles); nivel e o principal', 'Status': 'SENSIBILIDADE', 'Justificativa': 'Secao 46'},
    {'Item': 'Outcome per capita', 'Decisao': 'Nao disponivel nesta especificacao', 'Status': 'PENDENTE', 'Justificativa': 'Sem fonte populacional aprovada; secao 47 (nao essencial, nao bloqueia)'},
    {'Item': 'Sigilo', 'Decisao': 'Municipio 5003900 excluido do painel causal principal inteiro', 'Status': 'CONGELADO', 'Justificativa': 'Secao 48; nunca imputado nem zerado'},
    {'Item': 'Painel', 'Decisao': 'Balanceado, exigido na especificacao principal', 'Status': 'CONGELADO', 'Justificativa': 'Viavel apos exclusao do sigilo; secao 49'},
    {'Item': 'Covariaveis', 'Decisao': 'Nenhuma no principal; sensibilidade futura se houver dados aprovados', 'Status': 'CONGELADO', 'Justificativa': 'Gate D; secoes 42-44 (decisao em si e congelada; dado futuro e sensibilidade)'},
    {'Item': 'Spillover', 'Decisao': 'Sem filtro no principal; thresholds ja documentados (25/50/100km, arranjo) como sensibilidades futuras', 'Status': 'CONGELADO', 'Justificativa': 'Secao 54 (decisao em si e congelada; filtro e sensibilidade)'},
    {'Item': 'Estimando', 'Decisao': 'ATT(g,t) Callaway-Sant Anna, never-treated, PT incondicional, k=-2..+2 principal', 'Status': 'CONGELADO', 'Justificativa': 'Secao 55'},
    {'Item': 'Estimador futuro', 'Decisao': "Callaway-Sant Anna / DiD escalonado (nao executado nesta etapa)", 'Status': 'CONGELADO', 'Justificativa': 'CONTRATO_CAUSAL.md; secoes 56-59'},
])
display(estilizar_tabela_ipt(contrato_especificacao))

,Item,Decisao,Status,Justificativa
0,Tratamento,"Presenca operacional de campus Fase II (cadastro causal D11, nao redefinido)",CONGELADO,Definicao ja aprovada; secao 36
1,Coorte g,g = ano_coorte_candidata (campo canonico do cadastro),CONGELADO,Regra unica cobre os 129; secao 37
2,Ano zero,"k=0 no ano g; para Cabo Frio, g=2010 (primeiro ano completo)",CONGELADO,Secao 37 e 39
3,Evidencia institucional de antecipacao,"Insuficiente para regra geral (128/129 sob proxy_censo, sem transicao documentada)",LIMITACAO,CONTRATO_CAUSAL.md/PROTOCOLO_PRE_ANALISE.md 6.7; secao 38
4,Suposicao de antecipacao principal,"0 periodos (hipotese identificadora, nao fato observado)",CONGELADO,Padrao Callaway-Sant Anna; secao 38
5,Cabo Frio,"2009 = transicao, excluida da estimacao; 2010 = primeiro ano completo = g = k=0",CONGELADO,Regra ja no cadastro causal; secao 39
6,Tratados principais,129 candidatos (candidato_amostra_principal=True),CONGELADO,"D12/D13/D14, reproduzido; secao 37"
7,Controle principal,"Never-treated, pool estrutural 4.964 menos sigilo = 4.963",CONGELADO,Secao 40 e 48
8,Not-yet-treated,"Nao utilizado (escolha metodologica deliberada, nao proibicao tecnica)",CONGELADO,Cinco razoes explicitas; secao 41
9,Periodo total,2007-2019 (painel inteiro),CONGELADO,Nenhuma justificativa causal para cortar; secao 50


## 61. Regra de congelamento

`ESPECIFICACAO_CAUSAL_CONGELADA = SIM` somente se nenhum item **essencial**
estiver `PENDENTE` ou `BLOQUEADOR`. Itens essenciais: tratamento, coorte g,
ano zero, suposição de antecipação principal (não a evidência
institucional, que é registrada como limitação por natureza, não como algo
a resolver), controle principal, período total, outcome, sigilo,
covariáveis, estimando.

In [31]:
ITENS_ESSENCIAIS = [
    'Tratamento', 'Coorte g', 'Ano zero', 'Suposicao de antecipacao principal', 'Controle principal',
    'Periodo total', 'Outcome', 'Sigilo', 'Covariaveis', 'Estimando',
]
essenciais = contrato_especificacao.loc[contrato_especificacao['Item'].isin(ITENS_ESSENCIAIS)]
bloqueadores_d14 = essenciais.loc[essenciais['Status'].isin(['PENDENTE', 'BLOQUEADOR'])]

print(f"Itens essenciais verificados: {len(essenciais)}/{len(ITENS_ESSENCIAIS)}")
print(f"Itens essenciais pendentes/bloqueadores: {len(bloqueadores_d14)}")
if len(bloqueadores_d14):
    print(bloqueadores_d14[['Item', 'Status']].to_string(index=False))

ESPECIFICACAO_CAUSAL_CONGELADA = 'SIM' if bloqueadores_d14.empty else 'NAO'
PRONTO_PARA_CONSTRUIR_AMOSTRA_CAUSAL = 'SIM' if bloqueadores_d14.empty else 'NAO'
print()
print(f'ESPECIFICACAO_CAUSAL_CONGELADA = {ESPECIFICACAO_CAUSAL_CONGELADA}')
print(f'PRONTO_PARA_CONSTRUIR_AMOSTRA_CAUSAL = {PRONTO_PARA_CONSTRUIR_AMOSTRA_CAUSAL}')

item_pendente_nao_essencial = contrato_especificacao.loc[contrato_especificacao['Status'] == 'PENDENTE']
print()
print('Itens PENDENTE fora da lista essencial (nao bloqueiam o congelamento):')
print(item_pendente_nao_essencial[['Item', 'Decisao']].to_string(index=False) if len(item_pendente_nao_essencial) else '  nenhum')

item_limitacao = contrato_especificacao.loc[contrato_especificacao['Status'] == 'LIMITACAO']
print()
print('Itens LIMITACAO (registrados, nao bloqueiam -- distintos de PENDENTE/BLOQUEADOR):')
print(item_limitacao[['Item', 'Decisao']].to_string(index=False) if len(item_limitacao) else '  nenhum')

Itens essenciais verificados: 10/10
Itens essenciais pendentes/bloqueadores: 0

ESPECIFICACAO_CAUSAL_CONGELADA = SIM
PRONTO_PARA_CONSTRUIR_AMOSTRA_CAUSAL = SIM

Itens PENDENTE fora da lista essencial (nao bloqueiam o congelamento):
              Item                            Decisao
Outcome per capita Nao disponivel nesta especificacao

Itens LIMITACAO (registrados, nao bloqueiam -- distintos de PENDENTE/BLOQUEADOR):
                                  Item                                                                            Decisao
Evidencia institucional de antecipacao Insuficiente para regra geral (128/129 sob proxy_censo, sem transicao documentada)


## 62. Amostra causal final: definida logicamente, não materializada

Com a especificação congelada, as unidades que **entrariam** na amostra
causal ficam definidas logicamente pelas funções de
`src/define_especificacao_causal.py` — `unidades_tratadas_principal()`
(129) e `unidades_controle_principal()` (4.963). **Nenhuma nova amostra é
persistida em disco nesta etapa** — o painel D11 permanece intocado, e a
construção/persistência de um artefato de amostra causal (com `event_time`,
filtro de anos excluídos aplicado, etc.) fica para a etapa seguinte, que já
teria uma especificação congelada para seguir sem ambiguidade.

In [32]:
print(f"Unidades tratadas (logicamente definidas): {len(tratados_d14)}")
print(f"Unidades controle (logicamente definidas): {len(controles_d14)}")
print(f"Total logico da amostra causal candidata: {len(tratados_d14) + len(controles_d14)}")
print("Nenhum arquivo novo foi escrito com esta amostra.")

Unidades tratadas (logicamente definidas): 129
Unidades controle (logicamente definidas): 4963
Total logico da amostra causal candidata: 5092
Nenhum arquivo novo foi escrito com esta amostra.


## 63. Encerramento do D14 e próxima etapa

`ESPECIFICACAO_CAUSAL_CONGELADA = SIM`

`PRONTO_PARA_CONSTRUIR_AMOSTRA_CAUSAL = SIM`

`DESENHO_CAUSAL_APROVADO = NÃO` (congelar a especificação não é aprovar o
desenho — aprovação viria só depois de suporte comum/balanceamento e,
eventualmente, da própria estimação com diagnóstico de robustez).

`EVIDENCIA_INSTITUCIONAL_ANTECIPACAO = INSUFICIENTE_PARA_REGRA_GERAL`
(limitação, registrada — não bloqueia o congelamento).

`SUPOSICAO_ANTECIPACAO_PRINCIPAL = 0_PERIODOS` (hipótese identificadora da
especificação principal, não fato observado).

`GRUPO_COMPARACAO_PRINCIPAL = NEVER_TREATED`

`NOT_YET_TREATED = NÃO UTILIZADO` (escolha metodológica deliberada, seção
41 — não proibição técnica do cadastro).

**Itens registrados como sensibilidade, não como bloqueio**: janela 3 pré,
transformação log1p, covariáveis condicionais (X), filtros espaciais de
spillover. **Item pendente não essencial**: outcome per capita
(indisponível, sem nova coleta nesta tarefa). **Item de limitação, não
pendência**: evidência institucional de antecipação insuficiente para uma
regra geral (128/129 candidatos).

### Próxima etapa

**D15 — Construção da Amostra Causal Congelada.** Deverá materializar, sem
estimar nenhum efeito:

- os 129 tratados conforme o contrato (seção 60);
- os 4.963 controles never-treated;
- painel balanceado;
- período 2007-2019;
- exclusão do município `5003900` (sigilo);
- aplicação da exclusão institucional de 2009 para Cabo Frio
  (`anos_excluir_estimacao`);
- metadados necessários para a futura estimação (coorte, `k`,
  `event_time` calculado apenas quando a amostra for de fato
  materializada);
- validações da população resultante.

Nenhum estimador é executado no D15. Nenhuma dessas etapas é executada
neste notebook.